In [ ]:
# ============================================================
# CELL 1 — Import packages and set project paths
# GitHub-ready version: uses relative paths from the repository root
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Project folders
# ------------------------------------------------------------
# This cell works whether Jupyter is launched from the repository root
# or from the notebooks/ directory.
current_dir = Path.cwd().resolve()

if (current_dir / "data" / "raw").exists():
    PROJECT_DIR = current_dir
elif (current_dir.parent / "data" / "raw").exists():
    PROJECT_DIR = current_dir.parent
else:
    # Fallback: assume the notebook is in notebooks/ under the repo root.
    PROJECT_DIR = current_dir

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
TABLE_DIR = RESULTS_DIR / "tables"
FIGURE_DIR = RESULTS_DIR / "figures"
BERTOPIC_DIR = RESULTS_DIR / "bertopic"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
BERTOPIC_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Find Scopus CSV files
# ------------------------------------------------------------
# The repository includes one gzipped CSV file. The code also supports
# multiple uncompressed Scopus export_*.csv files placed in data/raw/.
csv_files = sorted(list(RAW_DIR.glob("export_*.csv")) + list(RAW_DIR.glob("export_*.csv.gz")))

print("Project directory")
print("-----------------")
print(PROJECT_DIR)
print()

print("Raw data directory")
print("------------------")
print(RAW_DIR)
print()

print(f"Number of CSV files found: {len(csv_files)}")
for path in csv_files:
    print(path.name)


In [ ]:
# ============================================================
# CELL 2 — Load and merge Scopus CSV files
# Supports both .csv and .csv.gz files
# ============================================================

if len(csv_files) == 0:
    raise FileNotFoundError(
        "No Scopus CSV files were found. "
        "Check whether the Scopus CSV files are located in the data/raw folder."
    )

df_list = []

for path in csv_files:
    temp = pd.read_csv(path, encoding="utf-8-sig", low_memory=False, compression="infer")
    temp["source_file"] = path.name
    df_list.append(temp)

df_raw = pd.concat(df_list, ignore_index=True)

print("Raw merged dataset")
print("------------------")
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]:,}")
print()
print("Columns:")
print(df_raw.columns.tolist())


In [ ]:
# ============================================================
# CELL 3 — Basic audit of raw merged dataset
# ============================================================

print("Year range")
print("----------")
print(df_raw["Year"].min(), "to", df_raw["Year"].max())
print()

print("Document type counts")
print("--------------------")
print(df_raw["Document Type"].value_counts(dropna=False))
print()

print("Number of source titles")
print("-----------------------")
print(df_raw["Source title"].nunique(dropna=True))
print()

print("Top source titles")
print("-----------------")
display(
    df_raw["Source title"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "Source title", "Source title": "No. of records"})
    .head(100)
)

In [ ]:
# ============================================================
# CELL 4 — Standardize source titles and apply final journal allowlist
# ============================================================

def normalize_source_title(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x.casefold()


# ------------------------------------------------------------
# Final veterinary journal allowlist
# Animal Welfare, Veterinary and Animal Science,
# and Veterinary Medicine: Research and Reports were excluded.
# ------------------------------------------------------------
source_title_map = {
    "veterinary journal": "Veterinary Journal",
    "the veterinary journal": "Veterinary Journal",
    "veterinary journal (london, england : 1997)": "Veterinary Journal",

    "veterinary sciences": "Veterinary Sciences",
    "irish veterinary journal": "Irish Veterinary Journal",

    "veterinary parasitology": "Veterinary Parasitology",
    "veterinary anaesthesia and analgesia": "Veterinary Anaesthesia and Analgesia",
    "journal of veterinary internal medicine": "Journal of Veterinary Internal Medicine",
    "equine veterinary journal": "Equine Veterinary Journal",
    "veterinary microbiology": "Veterinary Microbiology",
    "preventive veterinary medicine": "Preventive Veterinary Medicine",

    "frontiers in veterinary science": "Frontiers in Veterinary Science",
    "bmc veterinary research": "BMC Veterinary Research",
    "veterinary research": "Veterinary Research",
    "veterinary quarterly": "Veterinary Quarterly",
    "pakistan veterinary journal": "Pakistan Veterinary Journal",
}

final_journals = sorted(set(source_title_map.values()))

df_raw["source_title_norm"] = df_raw["Source title"].apply(normalize_source_title)
df_raw["source_title_clean"] = df_raw["source_title_norm"].map(source_title_map)

df_selected = df_raw[df_raw["source_title_clean"].isin(final_journals)].copy()

print("Selected veterinary journal dataset")
print("-----------------------------------")
print(f"Rows before duplicate removal: {df_selected.shape[0]:,}")
print(f"Selected journals: {df_selected['source_title_clean'].nunique()}")
print()

print("Journal counts")
print("--------------")
display(
    df_selected["source_title_clean"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="No. of records")
)

In [ ]:
# ============================================================
# CELL 5 — Check excluded source titles
# ============================================================

excluded_sources = (
    df_raw[df_raw["source_title_clean"].isna()]
    ["Source title"]
    .value_counts()
    .reset_index()
)

excluded_sources.columns = ["Excluded source title", "No. of records"]

print("Top excluded source titles")
print("--------------------------")
display(excluded_sources.head(50))

In [ ]:
# ============================================================
# CELL 6 — Apply year, document type, and text filters
# ============================================================

df_selected["Year"] = pd.to_numeric(df_selected["Year"], errors="coerce")
df_selected["Cited by"] = pd.to_numeric(df_selected["Cited by"], errors="coerce").fillna(0)

n_before_filter = len(df_selected)

df_filtered = df_selected[
    (df_selected["Year"] >= 2010)
    & (df_selected["Year"] <= 2025)
    & (df_selected["Document Type"].isin(["Article", "Review"]))
].copy()

n_after_basic_filter = len(df_filtered)

print("Filtering summary")
print("-----------------")
print(f"Before year/document-type filtering: {n_before_filter:,}")
print(f"After year/document-type filtering:  {n_after_basic_filter:,}")
print()

print("Document type counts")
print("--------------------")
print(df_filtered["Document Type"].value_counts())
print()

print("Year range")
print("----------")
print(df_filtered["Year"].min(), "to", df_filtered["Year"].max())

In [ ]:
# ============================================================
# CELL 7 — Safe duplicate removal using DOI and title
# ============================================================

def normalize_text_for_duplicate(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"[^\w\s]", "", x)
    return x


df_filtered["doi_clean"] = (
    df_filtered["DOI"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"nan": "", "none": ""})
)

df_filtered["title_clean"] = df_filtered["Title"].apply(normalize_text_for_duplicate)

n_before_dedup = len(df_filtered)

# ------------------------------------------------------------
# Step 1. Remove duplicated DOI records only when DOI is present
# ------------------------------------------------------------
df_with_doi = df_filtered[df_filtered["doi_clean"] != ""].copy()
df_without_doi = df_filtered[df_filtered["doi_clean"] == ""].copy()

df_with_doi = df_with_doi.drop_duplicates(subset=["doi_clean"], keep="first")

# ------------------------------------------------------------
# Step 2. Combine DOI and non-DOI records
# ------------------------------------------------------------
df_dedup = pd.concat([df_with_doi, df_without_doi], ignore_index=True)

# ------------------------------------------------------------
# Step 3. Remove duplicated titles as a secondary check
# ------------------------------------------------------------
df_dedup = df_dedup.drop_duplicates(subset=["title_clean"], keep="first")

n_after_dedup = len(df_dedup)

print("Duplicate removal summary")
print("-------------------------")
print(f"Before duplicate removal: {n_before_dedup:,}")
print(f"After duplicate removal:  {n_after_dedup:,}")
print(f"Removed records:          {n_before_dedup - n_after_dedup:,}")

In [ ]:
# ============================================================
# CELL 8 — Construct text fields for BERTopic
# ============================================================

def clean_text_field(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x


df_dedup["title_text"] = df_dedup["Title"].apply(clean_text_field)
df_dedup["abstract_text"] = df_dedup["Abstract"].apply(clean_text_field)

df_dedup["title_word_count"] = df_dedup["title_text"].apply(lambda x: len(x.split()))
df_dedup["abstract_word_count"] = df_dedup["abstract_text"].apply(lambda x: len(x.split()))

df_dedup["text_title_only"] = df_dedup["title_text"]
df_dedup["text_abstract_only"] = df_dedup["abstract_text"]
df_dedup["text_title_abstract"] = (
    df_dedup["title_text"] + ". " + df_dedup["abstract_text"]
).str.strip()

# ------------------------------------------------------------
# Define bibliometric dataset and BERTopic dataset separately
# ------------------------------------------------------------
df_biblio = df_dedup.copy()

df_bertopic = df_dedup[
    (df_dedup["title_word_count"] >= 3)
    & (df_dedup["abstract_word_count"] >= 20)
].copy()

print("Text-field summary")
print("------------------")
print(f"Final bibliometric dataset: {len(df_biblio):,}")
print(f"Final BERTopic dataset:     {len(df_bertopic):,}")
print(f"Excluded before BERTopic:   {len(df_biblio) - len(df_bertopic):,}")
print()

print("Title word count")
print("----------------")
print(df_biblio["title_word_count"].describe())
print()

print("Abstract word count")
print("-------------------")
print(df_biblio["abstract_word_count"].describe())

In [ ]:
# ============================================================
# CELL 9 — Create record flow table
# ============================================================

record_flow = pd.DataFrame(
    [
        {
            "Stage": "Initial Scopus export",
            "Description": "All downloaded Scopus CSV files merged",
            "No. of records": len(df_raw),
        },
        {
            "Stage": "After journal selection",
            "Description": "Records retained from the final veterinary journal allowlist",
            "No. of records": len(df_selected),
        },
        {
            "Stage": "After year and document-type filtering",
            "Description": "Articles and reviews published from 2010 to 2025",
            "No. of records": len(df_filtered),
        },
        {
            "Stage": "After duplicate removal",
            "Description": "Duplicate records removed using DOI and title information",
            "No. of records": len(df_biblio),
        },
        {
            "Stage": "Final bibliometric dataset",
            "Description": "Dataset used for publication, citation, country, and keyword analyses",
            "No. of records": len(df_biblio),
        },
        {
            "Stage": "Final BERTopic dataset",
            "Description": "Records with sufficient title and abstract text",
            "No. of records": len(df_bertopic),
        },
    ]
)

display(record_flow)

record_flow.to_csv(
    TABLE_DIR / "table_record_flow.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# CELL 10 — Create journal composition table with analysis period
# ============================================================

# ------------------------------------------------------------
# Ensure Year is integer
# ------------------------------------------------------------
df_biblio["Year"] = pd.to_numeric(df_biblio["Year"], errors="coerce").astype("Int64")

# ------------------------------------------------------------
# Document type count table
# ------------------------------------------------------------
journal_doc_counts = (
    df_biblio
    .groupby(["source_title_clean", "Document Type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["Article", "Review"]:
    if col not in journal_doc_counts.columns:
        journal_doc_counts[col] = 0

# ------------------------------------------------------------
# Analysis period by journal
# ------------------------------------------------------------
journal_period = (
    df_biblio
    .groupby("source_title_clean")
    .agg(
        start_year=("Year", "min"),
        end_year=("Year", "max")
    )
    .reset_index()
)

journal_period["Analysis period"] = (
    journal_period["start_year"].astype(str)
    + "–"
    + journal_period["end_year"].astype(str)
)

# ------------------------------------------------------------
# Merge document counts and analysis period
# ------------------------------------------------------------
journal_table = journal_doc_counts.merge(
    journal_period[["source_title_clean", "Analysis period"]],
    on="source_title_clean",
    how="left"
)

journal_table["Total documents"] = journal_table["Article"] + journal_table["Review"]

journal_table = journal_table.rename(columns={"source_title_clean": "Journal title"})

journal_table = journal_table[
    ["Journal title", "Analysis period", "Article", "Review", "Total documents"]
].sort_values("Total documents", ascending=False)

# ------------------------------------------------------------
# Add total row
# ------------------------------------------------------------
total_row = pd.DataFrame(
    [{
        "Journal title": "Total",
        "Analysis period": f"{int(df_biblio['Year'].min())}–{int(df_biblio['Year'].max())}",
        "Article": journal_table["Article"].sum(),
        "Review": journal_table["Review"].sum(),
        "Total documents": journal_table["Total documents"].sum(),
    }]
)

journal_table_final = pd.concat([journal_table, total_row], ignore_index=True)

display(journal_table_final)

journal_table_final.to_csv(
    TABLE_DIR / "table_journal_composition.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Journal composition table saved:")
print(TABLE_DIR / "table_journal_composition.csv")

In [ ]:
# ============================================================
# CELL 11 — Save cleaned datasets
# ============================================================

biblio_path = PROCESSED_DIR / "veterinary_science_bibliometric_dataset_2010_2025.csv"
bertopic_path = PROCESSED_DIR / "veterinary_science_bertopic_dataset_2010_2025.csv"

df_biblio.to_csv(biblio_path, index=False, encoding="utf-8-sig")
df_bertopic.to_csv(bertopic_path, index=False, encoding="utf-8-sig")

print("Saved files")
print("-----------")
print(biblio_path)
print(bertopic_path)
print()

print("Final dataset sizes")
print("-------------------")
print(f"Bibliometric dataset: {df_biblio.shape}")
print(f"BERTopic dataset:     {df_bertopic.shape}")

In [ ]:
# ============================================================
# CELL 12 — Annual publication and citation trend table
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# ------------------------------------------------------------
# Annual summary
# ------------------------------------------------------------
annual_summary = (
    df_biblio
    .groupby("Year")
    .agg(
        documents=("Title", "count"),
        total_citations=("Cited by", "sum"),
        mean_citations=("Cited by", "mean"),
        median_citations=("Cited by", "median")
    )
    .reset_index()
    .sort_values("Year")
)

annual_summary["Year"] = annual_summary["Year"].astype(int)
annual_summary["documents"] = annual_summary["documents"].astype(int)
annual_summary["total_citations"] = annual_summary["total_citations"].astype(int)
annual_summary["mean_citations"] = annual_summary["mean_citations"].round(2)
annual_summary["median_citations"] = annual_summary["median_citations"].round(2)

display(annual_summary)

annual_summary.to_csv(
    TABLE_DIR / "annual_publication_citation_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Annual summary saved:")
print(TABLE_DIR / "annual_publication_citation_summary.csv")

In [ ]:
# ============================================================
# CELL 13 — Figure 2. Annual publication and citation trends
# Rounded y-axis limits and fixed tick intervals
# ============================================================

import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MultipleLocator

plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.unicode_minus"] = False

# ------------------------------------------------------------
# Y-axis settings
# ------------------------------------------------------------
doc_tick_interval = 1000
citation_tick_interval = 10000

doc_max = annual_summary["documents"].max()
citation_max = annual_summary["total_citations"].max()

# ------------------------------------------------------------
# Manual y-axis maximum
# Set to None if you want automatic rounded limits
# ------------------------------------------------------------
doc_ymax_manual = 7000
citation_ymax_manual = 100000

doc_ymax_auto = math.ceil(doc_max / doc_tick_interval) * doc_tick_interval
citation_ymax_auto = math.ceil(citation_max / citation_tick_interval) * citation_tick_interval

doc_ymax = doc_ymax_manual if doc_ymax_manual is not None else doc_ymax_auto
citation_ymax = citation_ymax_manual if citation_ymax_manual is not None else citation_ymax_auto

# ------------------------------------------------------------
# Colors
# ------------------------------------------------------------
bar_color = "#F4C2D7"     # light pink
line_color = "#6D6D6D"    # gray

# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------
fig, ax1 = plt.subplots(figsize=(5.5, 4.0))

# ------------------------------------------------------------
# Left y-axis: number of documents
# ------------------------------------------------------------
ax1.bar(
    annual_summary["Year"],
    annual_summary["documents"],
    width=0.75,
    color=bar_color,
    alpha=1.0,
    label="Documents",
    zorder=2
)

ax1.set_xlabel("Publication year", fontsize=13, fontweight="bold")
ax1.set_ylabel("Number of documents", fontsize=13, fontweight="bold")

ax1.set_ylim(0, doc_ymax)
ax1.yaxis.set_major_locator(MultipleLocator(doc_tick_interval))
ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:,.0f}"))

ax1.tick_params(
    axis="both",
    labelsize=11,
    direction="out",
    width=0.9
)

# ------------------------------------------------------------
# Right y-axis: total citations
# ------------------------------------------------------------
ax2 = ax1.twinx()

ax2.plot(
    annual_summary["Year"],
    annual_summary["total_citations"],
    linewidth=1.2,
    color=line_color,
    label="Total citations",
    zorder=3
)

ax2.scatter(
    annual_summary["Year"],
    annual_summary["total_citations"],
    s=45,
    facecolors="none",
    edgecolors=line_color,
    linewidths=1.2,
    zorder=4
)

ax2.set_ylabel("Total citations", fontsize=13, fontweight="bold")

ax2.set_ylim(0, citation_ymax)
ax2.yaxis.set_major_locator(MultipleLocator(citation_tick_interval))
ax2.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:,.0f}"))

ax2.tick_params(
    axis="y",
    labelsize=11,
    direction="out",
    width=0.9
)

# ------------------------------------------------------------
# X-axis
# ------------------------------------------------------------
ax1.set_xlim(2009.4, 2025.6)
ax1.set_xticks(range(2010, 2026, 1))
ax1.set_xticklabels(
    range(2010, 2026, 1),
    rotation=45,
    ha="right"
)

# ------------------------------------------------------------
# Remove automatic margins
# ------------------------------------------------------------
ax1.margins(x=0, y=0)
ax2.margins(x=0, y=0)

# ------------------------------------------------------------
# Axis style
# ------------------------------------------------------------
for spine in ax1.spines.values():
    spine.set_linewidth(0.9)

for spine in ax2.spines.values():
    spine.set_linewidth(0.9)

# ------------------------------------------------------------
# Combined legend
# ------------------------------------------------------------
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

legend_handles = [
    Patch(
        facecolor=bar_color,
        edgecolor=bar_color,
        label="Documents"
    ),
    Line2D(
        [0], [0],
        color=line_color,
        linewidth=1.2,
        marker="o",
        markerfacecolor="none",
        markeredgecolor=line_color,
        markeredgewidth=1.2,
        label="Total citations"
    )
]

ax1.legend(
    handles=legend_handles,
    loc="upper right",
    frameon=False,
    fontsize=10
)

plt.tight_layout()

fig_path = RESULTS_DIR / "figure_2_annual_publication_citation_trends.png"
plt.savefig(fig_path, dpi=600, bbox_inches="tight")
plt.show()

print("Figure saved:")
print(fig_path)

print("Y-axis settings")
print("----------------")
print(f"Left y-axis max:  {doc_ymax:,}")
print(f"Left y interval:  {doc_tick_interval:,}")
print(f"Right y-axis max: {citation_ymax:,}")
print(f"Right y interval: {citation_tick_interval:,}")

In [ ]:
# ============================================================
# CELL 15 — Check affiliation-related columns
# ============================================================

# ------------------------------------------------------------
# Find possible affiliation columns
# ------------------------------------------------------------
affiliation_like_cols = [
    col for col in df_biblio.columns
    if "affil" in col.lower() or "country" in col.lower()
]

print("Affiliation- or country-related columns")
print("---------------------------------------")
for col in affiliation_like_cols:
    print(col)

print()
print("Sample affiliation records")
print("--------------------------")

for col in affiliation_like_cols:
    print(f"\nCOLUMN: {col}")
    display(
        df_biblio[[col]]
        .dropna()
        .head(5)
    )

In [ ]:
# ============================================================
# CELL 16 — Extract country information from affiliation fields
# ============================================================

import re
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Select affiliation column
# ------------------------------------------------------------
candidate_affiliation_cols = [
    "Affiliations",
    "Authors with affiliations",
    "Author full names",
]

affiliation_col = None

for col in candidate_affiliation_cols:
    if col in df_biblio.columns:
        affiliation_col = col
        break

if affiliation_col is None:
    raise ValueError(
        "No suitable affiliation column was found. "
        "Check the output of Cell 15 and set affiliation_col manually."
    )

print(f"Selected affiliation column: {affiliation_col}")

# ------------------------------------------------------------
# Country name cleaning dictionary
# ------------------------------------------------------------
country_replacements = {
    "united states": "United States",
    "usa": "United States",
    "u.s.a.": "United States",
    "u.s.": "United States",
    "united states of america": "United States",

    "uk": "United Kingdom",
    "u.k.": "United Kingdom",
    "england": "United Kingdom",
    "scotland": "United Kingdom",
    "wales": "United Kingdom",
    "northern ireland": "United Kingdom",

    "china": "China",
    "people's republic of china": "China",
    "pr china": "China",
    "p.r. china": "China",

    "hong kong": "Hong Kong",
    "taiwan": "Taiwan",
    "republic of korea": "South Korea",
    "korea": "South Korea",
    "south korea": "South Korea",

    "iran": "Iran",
    "islamic republic of iran": "Iran",

    "russian federation": "Russia",
    "russia": "Russia",

    "viet nam": "Vietnam",
    "vietnam": "Vietnam",

    "czech republic": "Czech Republic",
    "czechia": "Czech Republic",

    "turkiye": "Turkey",
    "türkiye": "Turkey",
    "turkey": "Turkey",

    "brasil": "Brazil",
    "brazil": "Brazil",

    "saudi arabia": "Saudi Arabia",
    "pakistan": "Pakistan",
    "india": "India",
    "japan": "Japan",
    "france": "France",
    "germany": "Germany",
    "italy": "Italy",
    "spain": "Spain",
    "netherlands": "Netherlands",
    "belgium": "Belgium",
    "switzerland": "Switzerland",
    "australia": "Australia",
    "canada": "Canada",
    "denmark": "Denmark",
    "sweden": "Sweden",
    "norway": "Norway",
    "finland": "Finland",
    "poland": "Poland",
    "egypt": "Egypt",
    "mexico": "Mexico",
    "argentina": "Argentina",
    "chile": "Chile",
    "thailand": "Thailand",
    "malaysia": "Malaysia",
    "indonesia": "Indonesia",
    "south africa": "South Africa",
    "new zealand": "New Zealand",
    "ireland": "Ireland",
    "portugal": "Portugal",
    "greece": "Greece",
    "austria": "Austria",
    "hungary": "Hungary",
}

# ------------------------------------------------------------
# Terms that are not countries and should be ignored
# ------------------------------------------------------------
non_country_terms = {
    "",
    "nan",
    "none",
    "null",
    "undefined",
    "university",
    "department",
    "college",
    "faculty",
    "institute",
    "school",
    "hospital",
    "center",
    "centre",
    "laboratory",
    "lab",
}

# ------------------------------------------------------------
# Function to clean a country token
# ------------------------------------------------------------
def clean_country_token(token):
    if pd.isna(token):
        return None

    token = str(token).strip()
    token = re.sub(r"\.$", "", token)
    token = re.sub(r"\s+", " ", token)
    token_lower = token.lower().strip()

    if token_lower in non_country_terms:
        return None

    if token_lower in country_replacements:
        return country_replacements[token_lower]

    # Remove postal-code-like strings
    if re.fullmatch(r"[0-9\- ]+", token_lower):
        return None

    # Remove tokens that are too long and likely not country names
    if len(token_lower.split()) > 5:
        return None

    # Keep title-case fallback
    return token.title()

# ------------------------------------------------------------
# Function to extract countries from Scopus affiliation text
# ------------------------------------------------------------
def extract_countries_from_affiliation(text):
    if pd.isna(text) or str(text).strip() == "":
        return []

    text = str(text)

    # Split multiple affiliations
    affiliation_parts = re.split(r";|\|", text)

    countries = []

    for part in affiliation_parts:
        part = part.strip()
        if part == "":
            continue

        # In Scopus affiliation strings, country usually appears after the last comma
        comma_parts = [p.strip() for p in part.split(",") if p.strip() != ""]

        if len(comma_parts) == 0:
            continue

        candidate = comma_parts[-1]
        country = clean_country_token(candidate)

        if country is not None:
            countries.append(country)

    # Unique countries per document
    countries = sorted(set(countries))

    return countries

# ------------------------------------------------------------
# Apply extraction
# ------------------------------------------------------------
df_biblio["countries"] = df_biblio[affiliation_col].apply(extract_countries_from_affiliation)
df_biblio["n_countries"] = df_biblio["countries"].apply(len)

# ------------------------------------------------------------
# Check extraction result
# ------------------------------------------------------------
print("Country extraction summary")
print("--------------------------")
print(f"Documents with at least one extracted country: {(df_biblio['n_countries'] > 0).sum():,}")
print(f"Documents without extracted country:          {(df_biblio['n_countries'] == 0).sum():,}")
print()

print("Sample extracted countries")
print("--------------------------")
display(
    df_biblio[[affiliation_col, "countries", "n_countries"]]
    .head(20)
)

In [ ]:
# ============================================================
# CELL 17 — Country contribution and international collaboration table
# ============================================================

# ------------------------------------------------------------
# Keep documents with country information
# ------------------------------------------------------------
df_country_docs = df_biblio[df_biblio["n_countries"] > 0].copy()

# ------------------------------------------------------------
# Long-format country table
# One country counted once per document
# ------------------------------------------------------------
country_rows = []

for idx, row in df_country_docs.iterrows():
    countries = row["countries"]
    n_countries = len(countries)

    if n_countries == 0:
        continue

    is_international = n_countries >= 2

    for country in countries:
        country_rows.append(
            {
                "document_index": idx,
                "Year": row["Year"],
                "Country": country,
                "n_countries_in_document": n_countries,
                "full_count": 1,
                "fractional_count": 1 / n_countries,
                "collaboration_type": "International" if is_international else "Single-country",
            }
        )

country_long = pd.DataFrame(country_rows)

# ------------------------------------------------------------
# Country summary
# ------------------------------------------------------------
country_summary = (
    country_long
    .groupby("Country")
    .agg(
        publications=("full_count", "sum"),
        contribution_score=("fractional_count", "sum"),
        international_documents=("collaboration_type", lambda x: (x == "International").sum()),
        single_country_documents=("collaboration_type", lambda x: (x == "Single-country").sum()),
    )
    .reset_index()
)

country_summary["international_collaboration_ratio"] = (
    country_summary["international_documents"]
    / country_summary["publications"]
)

country_summary["normalized_contribution_score"] = (
    country_summary["contribution_score"]
    / country_summary["contribution_score"].max()
)

country_summary = country_summary.sort_values(
    ["publications", "contribution_score"],
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# Display top countries
# ------------------------------------------------------------
display(country_summary.head(30))

# ------------------------------------------------------------
# Save tables
# ------------------------------------------------------------
country_long.to_csv(
    TABLE_DIR / "country_document_long_table.csv",
    index=False,
    encoding="utf-8-sig"
)

country_summary.to_csv(
    TABLE_DIR / "country_contribution_collaboration_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(TABLE_DIR / "country_document_long_table.csv")
print(TABLE_DIR / "country_contribution_collaboration_summary.csv")

In [ ]:
# ============================================================
# CELL 18A — Prepare world map for Figure 3
# ============================================================

import geopandas as gpd
import pandas as pd
import numpy as np
import re

# ------------------------------------------------------------
# Load world map
# ------------------------------------------------------------
try:
    world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
except Exception:
    world_url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
    world = gpd.read_file(world_url)

# ------------------------------------------------------------
# Standardize country-name column
# ------------------------------------------------------------
if "name" in world.columns:
    world = world.rename(columns={"name": "map_country"})
elif "ADMIN" in world.columns:
    world = world.rename(columns={"ADMIN": "map_country"})
elif "NAME" in world.columns:
    world = world.rename(columns={"NAME": "map_country"})
else:
    raise ValueError("Country-name column was not found in the world map dataset.")

world = world[world["map_country"] != "Antarctica"].copy()

# ------------------------------------------------------------
# Match country names between Scopus-derived countries and map
# ------------------------------------------------------------
map_name_corrections = {
    "United States": "United States of America",
    "South Korea": "South Korea",
    "North Korea": "North Korea",
    "United Kingdom": "United Kingdom",
    "Russia": "Russia",
    "Turkey": "Turkey",
    "Czech Republic": "Czechia",
    "Czechia": "Czechia",
    "Vietnam": "Vietnam",
    "Iran": "Iran",
    "United Arab Emirates": "United Arab Emirates",
    "Bosnia and Herzegovina": "Bosnia and Herzegovina",
    "Brunei": "Brunei",
    "Ivory Coast": "Côte d'Ivoire",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Democratic Republic of Congo": "Democratic Republic of the Congo",
    "Republic of Congo": "Republic of the Congo",
    "Tanzania": "United Republic of Tanzania",
    "Laos": "Laos",
    "Syria": "Syria",
    "Moldova": "Moldova",
    "Bolivia": "Bolivia",
    "Venezuela": "Venezuela",
}

map_df = country_summary.copy()

map_df["map_country"] = (
    map_df["Country"]
    .replace(map_name_corrections)
)

# ------------------------------------------------------------
# Add total collaboration documents
# ------------------------------------------------------------
map_df["total_collaboration_documents"] = (
    map_df["single_country_documents"]
    + map_df["international_documents"]
)

# ------------------------------------------------------------
# Merge with world map
# ------------------------------------------------------------
world_map = world.merge(
    map_df,
    how="left",
    on="map_country"
)

# ------------------------------------------------------------
# Check unmatched countries
# ------------------------------------------------------------
world_country_set = set(world["map_country"].dropna())
map_country_set = set(map_df["map_country"].dropna())

unmatched_countries = sorted(list(map_country_set - world_country_set))

print("World map loaded")
print("----------------")
print(f"World map countries: {world['map_country'].nunique():,}")
print(f"Countries in country_summary: {country_summary['Country'].nunique():,}")
print(f"Unmatched countries: {len(unmatched_countries):,}")
print(unmatched_countries[:50])

display(
    world_map[
        [
            "map_country",
            "publications",
            "contribution_score",
            "normalized_contribution_score",
            "international_collaboration_ratio"
        ]
    ]
    .dropna(subset=["publications"])
    .sort_values("publications", ascending=False)
    .head(20)
)

In [ ]:
# ============================================================
# CELL 18B-1 — Figure 3A. Publications and contribution score
# Vertical bar plot with map background
# Revised: label and colorbar placed horizontally on one line
# ============================================================

import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MultipleLocator
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from matplotlib import font_manager

# ------------------------------------------------------------
# Font check
# ------------------------------------------------------------
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
print("Arial installed:", "Arial" in available_fonts)

plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.unicode_minus"] = False

print("Current font.family setting:", plt.rcParams["font.family"])

# ------------------------------------------------------------
# Data
# ------------------------------------------------------------
top_n = 20

plot_a = (
    country_summary
    .sort_values("publications", ascending=False)
    .head(top_n)
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Axis settings
# ------------------------------------------------------------
pub_tick_interval = 2000

pub_ymax = math.ceil(
    max(plot_a["publications"].max(), plot_a["contribution_score"].max())
    / pub_tick_interval
) * pub_tick_interval

pub_ymax_manual = None
if pub_ymax_manual is not None:
    pub_ymax = pub_ymax_manual

# ------------------------------------------------------------
# Colors
# ------------------------------------------------------------
publication_bar_color = "#E6B8CF"
contribution_bar_color = "#D9D9D9"

cmap_contribution = LinearSegmentedColormap.from_list(
    "lightgray_to_pink",
    ["#F2F2F2", "#E6B8CF", "#D75F93"]
)

# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.set_zorder(2)
ax.patch.set_alpha(0)

# ------------------------------------------------------------
# Background map
# ------------------------------------------------------------
map_pos = [0.01, 0.04, 0.98, 0.98]

axins = ax.inset_axes(map_pos, zorder=0)

world_map.plot(
    column="normalized_contribution_score",
    cmap=cmap_contribution,
    vmin=0,
    vmax=1,
    linewidth=0.25,
    edgecolor="#D0D0D0",
    missing_kwds={"color": "#F7F7F7"},
    ax=axins
)

axins.set_xlim(-170, 170)
axins.set_ylim(-58, 85)
axins.set_aspect("equal", adjustable="box")
axins.margins(0)
axins.set_axis_off()

# ------------------------------------------------------------
# Colorbar — label and score bar on the same horizontal line
# ------------------------------------------------------------
# label: left side
ax.text(
    0.55,
    0.945,
    "Normalized contribution score",
    transform=ax.transAxes,
    ha="right",
    va="center",
    fontsize=8.5
)

# score bar: right side, same vertical position as label
# [left, bottom, width, height]
cax = ax.inset_axes([0.565, 0.945, 0.20, 0.020], zorder=10)
cax.patch.set_alpha(0)

sm = ScalarMappable(
    norm=Normalize(vmin=0, vmax=1),
    cmap=cmap_contribution
)
sm.set_array([])

cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
cb.set_ticks([0, 1])

cb.ax.tick_params(
    labelsize=7.5,
    length=2,
    pad=1,
    width=0.4,
    direction="out"
)

cb.outline.set_linewidth(0.4)

# ------------------------------------------------------------
# Foreground bars
# ------------------------------------------------------------
x = np.arange(len(plot_a))
bar_width = 0.42

ax.bar(
    x - bar_width / 2,
    plot_a["publications"],
    width=bar_width,
    color=publication_bar_color,
    edgecolor="none",
    label="Publications",
    zorder=5
)

ax.bar(
    x + bar_width / 2,
    plot_a["contribution_score"],
    width=bar_width,
    color=contribution_bar_color,
    edgecolor="none",
    label="Contribution score",
    zorder=5
)

# ------------------------------------------------------------
# Axis settings
# ------------------------------------------------------------
ax.set_ylabel(
    "Number of documents",
    fontsize=13,
    fontweight="bold"
)

ax.set_ylim(0, pub_ymax)
ax.yaxis.set_major_locator(MultipleLocator(pub_tick_interval))
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f"{y:,.0f}"))

ax.set_xlim(-0.6, len(plot_a) - 0.4)
ax.set_xticks(x)

ax.set_xticklabels(
    plot_a["Country"],
    rotation=90,
    ha="center",
    va="top",
    fontsize=11
)

ax.tick_params(
    axis="y",
    labelsize=11,
    direction="out",
    width=0.9,
    length=4
)

ax.tick_params(
    axis="x",
    labelsize=11,
    direction="out",
    width=0.9,
    length=4
)

ax.margins(x=0, y=0)

# ------------------------------------------------------------
# Legend
# ------------------------------------------------------------
ax.legend(
    frameon=False,
    fontsize=10,
    loc="upper center",
    bbox_to_anchor=(0.50, 1.12),
    ncol=2,
    columnspacing=1.2,
    handlelength=1.5
)

# ------------------------------------------------------------
# Spine style
# ------------------------------------------------------------
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.9)

# ------------------------------------------------------------
# Draw first, then check actual fonts
# ------------------------------------------------------------
plt.tight_layout()
fig.canvas.draw()

y_tick_fonts = sorted({
    label.get_fontproperties().get_name()
    for label in ax.get_yticklabels()
})

x_tick_fonts = sorted({
    label.get_fontproperties().get_name()
    for label in ax.get_xticklabels()
})

axis_label_fonts = [
    ax.yaxis.label.get_fontproperties().get_name()
]

print("Actual y-axis tick label font(s):", y_tick_fonts)
print("Actual x-axis tick label font(s):", x_tick_fonts)
print("Actual axis label font(s):", axis_label_fonts)

fig_path = RESULTS_DIR / "figure_3a_country_contribution_map_background.png"
plt.savefig(fig_path, dpi=600, bbox_inches="tight")
plt.show()

print("Figure saved:")
print(fig_path)

In [ ]:
# ============================================================
# CELL 18B-2 — Figure 3B. Single-country vs international collaboration
# Vertical stacked bar plot with map background
# Revised: label and colorbar placed horizontally on one line
# ============================================================

import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MultipleLocator
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from matplotlib import font_manager

# ------------------------------------------------------------
# Font check
# ------------------------------------------------------------
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
print("Arial installed:", "Arial" in available_fonts)

plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.unicode_minus"] = False

print("Current font.family setting:", plt.rcParams["font.family"])

# ------------------------------------------------------------
# Data
# ------------------------------------------------------------
top_n = 20

plot_b = (
    country_summary
    .sort_values("publications", ascending=False)
    .head(top_n)
    .copy()
    .reset_index(drop=True)
)

plot_b["total_collaboration_documents"] = (
    plot_b["single_country_documents"]
    + plot_b["international_documents"]
)

# ------------------------------------------------------------
# Axis settings
# ------------------------------------------------------------
collab_tick_interval = 2000

collab_ymax = math.ceil(
    plot_b["total_collaboration_documents"].max()
    / collab_tick_interval
) * collab_tick_interval

collab_ymax_manual = None
if collab_ymax_manual is not None:
    collab_ymax = collab_ymax_manual

# ------------------------------------------------------------
# Colors
# ------------------------------------------------------------
single_country_bar_color = "#FFE0A1"
international_bar_color = "#AAFF64"

cmap_collaboration = LinearSegmentedColormap.from_list(
    "lightgray_to_pink",
    ["#FFE0A1", "#FFFFFF", "#AAFF64"]
)

# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.set_zorder(2)
ax.patch.set_alpha(0)

# ------------------------------------------------------------
# Background map
# ------------------------------------------------------------
map_pos = [0.01, 0.04, 0.98, 0.98]

axins = ax.inset_axes(map_pos, zorder=0)

world_map.plot(
    column="international_collaboration_ratio",
    cmap=cmap_collaboration,
    vmin=0,
    vmax=1,
    linewidth=0.25,
    edgecolor="#D0D0D0",
    missing_kwds={"color": "#F7F7F7"},
    ax=axins
)

axins.set_xlim(-170, 170)
axins.set_ylim(-58, 85)
axins.set_aspect("equal", adjustable="box")
axins.margins(0)
axins.set_axis_off()

# ------------------------------------------------------------
# Colorbar — label and score bar on the same horizontal line
# ------------------------------------------------------------
# label: left side
ax.text(
    0.55,
    0.945,
    "International collaboration ratio",
    transform=ax.transAxes,
    ha="right",
    va="center",
    fontsize=8.5
)

# score bar: right side, same vertical position as label
# [left, bottom, width, height]
cax = ax.inset_axes([0.565, 0.945, 0.20, 0.020], zorder=10)
cax.patch.set_alpha(0)

sm = ScalarMappable(
    norm=Normalize(vmin=0, vmax=1),
    cmap=cmap_collaboration
)
sm.set_array([])

cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
cb.set_ticks([0, 1])

cb.ax.tick_params(
    labelsize=7.5,
    length=2,
    pad=1,
    width=0.4,
    direction="out"
)

cb.outline.set_linewidth(0.4)

# ------------------------------------------------------------
# Foreground stacked bars
# ------------------------------------------------------------
x = np.arange(len(plot_b))
bar_width = 0.72

ax.bar(
    x,
    plot_b["single_country_documents"],
    width=bar_width,
    color=single_country_bar_color,
    edgecolor="none",
    label="Single-country",
    zorder=5
)

ax.bar(
    x,
    plot_b["international_documents"],
    bottom=plot_b["single_country_documents"],
    width=bar_width,
    color=international_bar_color,
    edgecolor="none",
    label="International collaboration",
    zorder=5
)

# ------------------------------------------------------------
# Axis settings
# ------------------------------------------------------------
ax.set_ylabel(
    "Number of documents",
    fontsize=13,
    fontweight="bold"
)

ax.set_ylim(0, collab_ymax)
ax.yaxis.set_major_locator(MultipleLocator(collab_tick_interval))
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f"{y:,.0f}"))

ax.set_xlim(-0.6, len(plot_b) - 0.4)
ax.set_xticks(x)

ax.set_xticklabels(
    plot_b["Country"],
    rotation=90,
    ha="center",
    va="top",
    fontsize=11
)

ax.tick_params(
    axis="y",
    labelsize=11,
    direction="out",
    width=0.9,
    length=4
)

ax.tick_params(
    axis="x",
    labelsize=11,
    direction="out",
    width=0.9,
    length=4
)

ax.margins(x=0, y=0)

# ------------------------------------------------------------
# Legend
# ------------------------------------------------------------
ax.legend(
    frameon=False,
    fontsize=10,
    loc="upper center",
    bbox_to_anchor=(0.50, 1.12),
    ncol=2,
    columnspacing=1.2,
    handlelength=1.5
)

# ------------------------------------------------------------
# Spine style
# ------------------------------------------------------------
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.9)

# ------------------------------------------------------------
# Draw first, then check actual fonts
# ------------------------------------------------------------
plt.tight_layout()
fig.canvas.draw()

y_tick_fonts = sorted({
    label.get_fontproperties().get_name()
    for label in ax.get_yticklabels()
})

x_tick_fonts = sorted({
    label.get_fontproperties().get_name()
    for label in ax.get_xticklabels()
})

axis_label_fonts = [
    ax.yaxis.label.get_fontproperties().get_name()
]

print("Actual y-axis tick label font(s):", y_tick_fonts)
print("Actual x-axis tick label font(s):", x_tick_fonts)
print("Actual axis label font(s):", axis_label_fonts)

fig_path = RESULTS_DIR / "figure_3b_country_collaboration_map_background.png"
plt.savefig(fig_path, dpi=600, bbox_inches="tight")
plt.show()

print("Figure saved:")
print(fig_path)

In [ ]:
# ============================================================
# CELL 19 — Check keyword-related columns
# ============================================================

# ------------------------------------------------------------
# Find keyword-related columns
# ------------------------------------------------------------
keyword_like_cols = [
    col for col in df_biblio.columns
    if "keyword" in col.lower() or "index" in col.lower()
]

print("Keyword-related columns")
print("-----------------------")
for col in keyword_like_cols:
    print(col)

print()
print("Sample keyword records")
print("----------------------")

for col in keyword_like_cols:
    print(f"\nCOLUMN: {col}")
    display(
        df_biblio[[col]]
        .dropna()
        .head(10)
    )

In [ ]:
# ============================================================
# CELL 20 — Extract raw keywords and inspect raw keyword frequency
# ============================================================

import re
from collections import Counter

# ------------------------------------------------------------
# Select keyword columns
# ------------------------------------------------------------
keyword_cols = []

for col in ["Author Keywords", "Index Keywords"]:
    if col in df_biblio.columns:
        keyword_cols.append(col)

if len(keyword_cols) == 0:
    raise ValueError(
        "No keyword columns were found. "
        "Check Cell 19 output and set keyword_cols manually."
    )

print("Selected keyword columns:")
print(keyword_cols)

# ------------------------------------------------------------
# Basic keyword splitter
# ------------------------------------------------------------
def split_keywords(text):
    if pd.isna(text) or str(text).strip() == "":
        return []

    text = str(text)

    # Scopus keywords are usually separated by semicolons
    parts = re.split(r";|\|", text)

    cleaned = []
    for part in parts:
        kw = str(part).strip()
        kw = re.sub(r"\s+", " ", kw)
        kw = kw.strip(" ,.;:")
        if kw != "":
            cleaned.append(kw)

    return cleaned

# ------------------------------------------------------------
# Combine author and indexed keywords
# ------------------------------------------------------------
def get_raw_keywords(row):
    keywords = []

    for col in keyword_cols:
        keywords.extend(split_keywords(row.get(col, "")))

    # Remove duplicates within each document while preserving order
    seen = set()
    unique_keywords = []

    for kw in keywords:
        key = kw.casefold()
        if key not in seen:
            seen.add(key)
            unique_keywords.append(kw)

    return unique_keywords

df_biblio["raw_keywords"] = df_biblio.apply(get_raw_keywords, axis=1)
df_biblio["n_raw_keywords"] = df_biblio["raw_keywords"].apply(len)

# ------------------------------------------------------------
# Raw keyword frequency
# ------------------------------------------------------------
raw_counter = Counter()

for kws in df_biblio["raw_keywords"]:
    for kw in kws:
        raw_counter[kw] += 1

raw_keyword_freq = (
    pd.DataFrame(raw_counter.items(), columns=["raw_keyword", "frequency"])
    .sort_values("frequency", ascending=False)
    .reset_index(drop=True)
)

display(raw_keyword_freq.head(100))

raw_keyword_freq.to_csv(
    TABLE_DIR / "raw_keyword_frequency_top_terms.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Keyword coverage")
print("----------------")
print(f"Documents with at least one keyword: {(df_biblio['n_raw_keywords'] > 0).sum():,}")
print(f"Documents without keywords:          {(df_biblio['n_raw_keywords'] == 0).sum():,}")
print()
print("Saved:")
print(TABLE_DIR / "raw_keyword_frequency_top_terms.csv")

In [ ]:
# ============================================================
# CELL 21 — Clean and normalize keywords
# Revised for veterinary science corpus
# Additional removal of overly broad indexing/method terms
# ============================================================

import re
from collections import Counter

# ------------------------------------------------------------
# Keyword normalization dictionary
# ------------------------------------------------------------
keyword_normalization = {
    # --------------------------------------------------------
    # Veterinary public health / One Health
    # --------------------------------------------------------
    "one health": "One Health",
    "zoonosis": "zoonosis",
    "zoonoses": "zoonosis",
    "zoonotic disease": "zoonotic disease",
    "zoonotic diseases": "zoonotic disease",
    "veterinary epidemiology": "veterinary epidemiology",
    "veterinary public health": "veterinary public health",

    # --------------------------------------------------------
    # Disease terms
    # --------------------------------------------------------
    "disease": "disease",
    "diseases": "disease",
    "animal disease": "animal disease",
    "animal diseases": "animal disease",

    "dog disease": "dog disease",
    "dog diseases": "dog disease",
    "canine disease": "dog disease",
    "canine diseases": "dog disease",

    "cat disease": "cat disease",
    "cat diseases": "cat disease",
    "feline disease": "cat disease",
    "feline diseases": "cat disease",

    "horse disease": "horse disease",
    "horse diseases": "horse disease",
    "equine disease": "horse disease",
    "equine diseases": "horse disease",

    "cattle disease": "cattle disease",
    "cattle diseases": "cattle disease",
    "bovine disease": "cattle disease",
    "bovine diseases": "cattle disease",

    "pig disease": "pig disease",
    "pig diseases": "pig disease",
    "swine disease": "pig disease",
    "swine diseases": "pig disease",
    "porcine disease": "pig disease",
    "porcine diseases": "pig disease",

    "poultry disease": "poultry disease",
    "poultry diseases": "poultry disease",
    "avian disease": "poultry disease",
    "avian diseases": "poultry disease",

    # --------------------------------------------------------
    # Infectious disease / microbiology
    # --------------------------------------------------------
    "infection": "infection",
    "infections": "infection",
    "infectious disease": "infectious disease",
    "infectious diseases": "infectious disease",
    "bacterial infection": "bacterial infection",
    "bacterial infections": "bacterial infection",
    "viral infection": "viral infection",
    "viral infections": "viral infection",
    "parasitic infection": "parasitic infection",
    "parasitic infections": "parasitic infection",

    # --------------------------------------------------------
    # Antimicrobial resistance
    # --------------------------------------------------------
    "antimicrobial resistance": "antimicrobial resistance",
    "antibiotic resistance": "antimicrobial resistance",
    "drug resistance": "antimicrobial resistance",
    "antibacterial resistance": "antimicrobial resistance",
    "antimicrobial susceptibility": "antimicrobial susceptibility",
    "antibiotic susceptibility": "antimicrobial susceptibility",
    "multidrug resistance": "multidrug resistance",
    "multi-drug resistance": "multidrug resistance",
    "mdr": "multidrug resistance",

    # --------------------------------------------------------
    # Diagnosis / epidemiology
    # --------------------------------------------------------
    "diagnosis": "diagnosis",
    "diagnostic": "diagnosis",
    "diagnostics": "diagnosis",
    "molecular diagnosis": "molecular diagnosis",
    "epidemiology": "epidemiology",
    "prevalence": "prevalence",
    "incidence": "incidence",
    "risk factor": "risk factor",
    "risk factors": "risk factor",
    "surveillance": "surveillance",

    # --------------------------------------------------------
    # Laboratory / diagnostic methods
    # --------------------------------------------------------
    "polymerase chain reaction": "PCR",
    "pcr": "PCR",
    "real time polymerase chain reaction": "real-time PCR",
    "real-time polymerase chain reaction": "real-time PCR",
    "real-time pcr": "real-time PCR",
    "rt-pcr": "RT-PCR",
    "reverse transcription polymerase chain reaction": "RT-PCR",
    "enzyme linked immunosorbent assay": "ELISA",
    "enzyme-linked immunosorbent assay": "ELISA",
    "elisa": "ELISA",
    "histopathology": "histopathology",
    "histology": "histopathology",

    # --------------------------------------------------------
    # Immune and host response
    # --------------------------------------------------------
    "immune response": "immune response",
    "host immune response": "immune response",
    "inflammation": "inflammation",
    "inflammatory response": "inflammation",
    "oxidative stress": "oxidative stress",
    "antioxidant": "antioxidant response",
    "antioxidants": "antioxidant response",
    "antioxidant activity": "antioxidant response",
    "gene expression": "gene expression",
    "protein expression": "protein expression",
    "genotype": "genotype",
    "phylogeny": "phylogeny",

    # --------------------------------------------------------
    # Animal species
    # --------------------------------------------------------
    "dog": "dog",
    "dogs": "dog",
    "canine": "dog",
    "canis familiaris": "dog",

    "cat": "cat",
    "cats": "cat",
    "feline": "cat",
    "felis catus": "cat",

    "cattle": "cattle",
    "cow": "cattle",
    "cows": "cattle",
    "bovine": "cattle",
    "bos": "cattle",
    "bos taurus": "cattle",
    "bovinae": "cattle",

    "sheep": "sheep",
    "ovine": "sheep",
    "ovis aries": "sheep",

    "goat": "goat",
    "goats": "goat",
    "caprine": "goat",
    "capra hircus": "goat",

    "pig": "pig",
    "pigs": "pig",
    "swine": "pig",
    "porcine": "pig",
    "sus scrofa": "pig",
    "suidae": "pig",

    "horse": "horse",
    "horses": "horse",
    "equine": "horse",
    "equidae": "horse",
    "equus caballus": "horse",

    "poultry": "poultry",
    "chicken": "poultry",
    "chickens": "poultry",
    "broiler": "poultry",
    "broilers": "poultry",
    "avian": "poultry",

    # --------------------------------------------------------
    # Common pathogens
    # --------------------------------------------------------
    "escherichia coli": "Escherichia coli",
    "e. coli": "Escherichia coli",
    "salmonella": "Salmonella",
    "salmonella spp.": "Salmonella",
    "staphylococcus aureus": "Staphylococcus aureus",
    "s. aureus": "Staphylococcus aureus",
    "campylobacter": "Campylobacter",
    "campylobacter jejuni": "Campylobacter jejuni",
    "brucella": "Brucella",
    "mycobacterium bovis": "Mycobacterium bovis",
    "influenza": "influenza",
    "avian influenza": "avian influenza",
    "rabies": "rabies",
    "coronavirus": "coronavirus",
    "sars-cov-2": "SARS-CoV-2",
    "porcine reproductive and respiratory syndrome virus": "PRRSV",
    "prrsv": "PRRSV",

    # --------------------------------------------------------
    # Clinical / treatment terms
    # --------------------------------------------------------
    "treatment": "treatment",
    "therapy": "treatment",
    "therapeutics": "treatment",
    "surgery": "surgery",
    "surgical": "surgery",
    "anesthesia": "anesthesia",
    "anaesthesia": "anesthesia",
    "analgesia": "analgesia",
    "pain": "pain",
    "vaccine": "vaccination",
    "vaccines": "vaccination",
    "vaccination": "vaccination",
}

# ------------------------------------------------------------
# Exclusion list
# Remove broad Scopus/Emtree indexing terms and non-informative terms
# ------------------------------------------------------------
keyword_exclusion = {
    "",
    "article",
    "review",
    "study",
    "studies",
    "research",
    "analysis",
    "method",
    "methods",
    "result",
    "results",
    "effect",
    "effects",

    # overly broad field terms
    "veterinary",
    "veterinary medicine",
    "veterinary science",

    # overly broad animal / indexing terms
    "animal",
    "animals",
    "animalia",
    "animal experiment",
    "animal experiments",
    "animal model",
    "animal models",
    "animal tissue",
    "animal tissues",
    "animal cell",
    "animal cells",
    "nonhuman",
    "non-human",
    "controlled study",
    "priority journal",

    # human demographic/indexing terms
    "human",
    "humans",
    "male",
    "female",
    "adult",
    "aged",
    "young adult",
    "middle aged",
    "child",
    "infant",

    # broad biomedical or indexing terms
    "unclassified drug",
    "blood",
    "blood sampling",
    "body weight",
    "procedures",
    "procedure",
    "physiology",
    "metabolism",
    "genetics",
    "immunology",
    "microbiology",
    "parasitology",
    "virology",
    "pathology",
    "classification",

    # study design terms
    "retrospective study",
    "prospective study",
    "comparative study",
    "cross-sectional study",
    "case report",
    "case reports",

    # broad sample/laboratory terms
    "feces",
    "faeces",
    "fecal sample",
    "faecal sample",
    "isolation and purification",
    "nucleotide sequence",
    "amino acid sequence",
    "sequence analysis",
    "gene sequence",

    # broad drug indexing terms
    "drug effect",
    "drug effects",
    "drug therapy",
    "drug administration",
}

# ------------------------------------------------------------
# Cleaning function
# ------------------------------------------------------------
def normalize_keyword(kw):
    if pd.isna(kw):
        return None

    kw = str(kw).strip()
    kw = re.sub(r"\s+", " ", kw)
    kw = kw.strip(" ,.;:")

    if kw == "":
        return None

    key = kw.casefold()
    key = re.sub(r"^[\W_]+|[\W_]+$", "", key)
    key = re.sub(r"\s+", " ", key)

    if key in keyword_exclusion:
        return None

    if key in keyword_normalization:
        return keyword_normalization[key]

    # Basic singular handling
    if len(key) > 4 and key.endswith("ies"):
        candidate = key[:-3] + "y"
        if candidate in keyword_normalization:
            return keyword_normalization[candidate]
        if candidate in keyword_exclusion:
            return None

    if len(key) > 4 and key.endswith("s"):
        candidate = key[:-1]
        if candidate in keyword_normalization:
            return keyword_normalization[candidate]
        if candidate in keyword_exclusion:
            return None

    return key

# ------------------------------------------------------------
# Apply normalization
# ------------------------------------------------------------
def get_clean_keywords(raw_keywords):
    clean_keywords = []

    for kw in raw_keywords:
        clean_kw = normalize_keyword(kw)
        if clean_kw is not None:
            clean_keywords.append(clean_kw)

    # Remove duplicates within each document
    seen = set()
    unique_keywords = []

    for kw in clean_keywords:
        key = kw.casefold()
        if key not in seen:
            seen.add(key)
            unique_keywords.append(kw)

    return unique_keywords

df_biblio["clean_keywords"] = df_biblio["raw_keywords"].apply(get_clean_keywords)
df_biblio["n_clean_keywords"] = df_biblio["clean_keywords"].apply(len)

# ------------------------------------------------------------
# Check cleaned keyword frequency
# ------------------------------------------------------------
clean_counter = Counter()

for kws in df_biblio["clean_keywords"]:
    for kw in kws:
        clean_counter[kw] += 1

clean_keyword_freq = (
    pd.DataFrame(clean_counter.items(), columns=["keyword", "frequency"])
    .sort_values("frequency", ascending=False)
    .reset_index(drop=True)
)

display(clean_keyword_freq.head(100))

clean_keyword_freq.to_csv(
    TABLE_DIR / "clean_keyword_frequency_top_terms.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Cleaned keyword coverage")
print("------------------------")
print(f"Documents with at least one cleaned keyword: {(df_biblio['n_clean_keywords'] > 0).sum():,}")
print(f"Documents without cleaned keywords:          {(df_biblio['n_clean_keywords'] == 0).sum():,}")
print()
print("Saved:")
print(TABLE_DIR / "clean_keyword_frequency_top_terms.csv")

In [ ]:
# ============================================================
# CELL 22 — Final keyword refinement and Table 2 generation
# Revised to ensure that Table 2 reflects Section 2.4 preprocessing
# ============================================================

import ast
import re
import pandas as pd
import numpy as np
from collections import Counter

# ------------------------------------------------------------
# Helper: make sure keyword values are list-like
# ------------------------------------------------------------
def ensure_keyword_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    if isinstance(value, str):
        value = value.strip()

        if value == "":
            return []

        if value.startswith("[") and value.endswith("]"):
            try:
                parsed = ast.literal_eval(value)
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                pass

        return [v.strip() for v in re.split(r";|\|", value) if v.strip() != ""]

    return []


# ------------------------------------------------------------
# Final keyword normalization
# These corrections are applied before Table 2 generation
# ------------------------------------------------------------
final_keyword_normalization = {
    # Broken or variant keyword forms
    "apoultry": "poultry",
    "a poultry": "poultry",

    "bacteria (microorganisms": "bacteria",
    "bacteria (microorganism": "bacteria",
    "bacteria (microorganisms)": "bacteria",
    "bacteria (microorganism)": "bacteria",
    "bacterium": "bacteria",
    "bacteria": "bacteria",

    # Keep informative molecular terms
    "protein expression": "protein expression",
    "gene expression": "gene expression",

    # Diagnostic accuracy term
    "sensitivity and specificity": "sensitivity and specificity",
}


# ------------------------------------------------------------
# Final keyword exclusion
# These terms are too broad, indexing-driven, or noninformative
# ------------------------------------------------------------
final_keyword_exclusion = {
    "",
    "molecular sequence data",
    "in vitro study",
    "in vitro studies",
    "dna extraction",
    "DNA extraction",
}


# ------------------------------------------------------------
# Refinement function
# ------------------------------------------------------------
def refine_clean_keyword(kw):
    if pd.isna(kw):
        return None

    kw = str(kw).strip()
    kw = re.sub(r"\s+", " ", kw)
    kw = kw.strip(" ,.;:")

    if kw == "":
        return None

    key = kw.casefold()
    key = re.sub(r"\s+", " ", key).strip()

    exclusion_keys = {x.casefold() for x in final_keyword_exclusion}

    if key in exclusion_keys:
        return None

    if key in final_keyword_normalization:
        normalized = final_keyword_normalization[key]

        if normalized.casefold() in exclusion_keys:
            return None

        return normalized

    return kw


# ------------------------------------------------------------
# Apply final refinement to clean_keywords
# ------------------------------------------------------------
def refine_keyword_list(keyword_list):
    keyword_list = ensure_keyword_list(keyword_list)

    refined = []

    for kw in keyword_list:
        new_kw = refine_clean_keyword(kw)

        if new_kw is not None:
            refined.append(new_kw)

    # Remove duplicates within each document
    seen = set()
    unique_refined = []

    for kw in refined:
        key = kw.casefold()

        if key not in seen:
            seen.add(key)
            unique_refined.append(kw)

    return unique_refined


if "clean_keywords" not in df_biblio.columns:
    raise ValueError(
        "The column 'clean_keywords' was not found. "
        "Please run Cell 21 before running this cell."
    )

df_biblio["clean_keywords"] = df_biblio["clean_keywords"].apply(refine_keyword_list)
df_biblio["n_clean_keywords"] = df_biblio["clean_keywords"].apply(len)


# ------------------------------------------------------------
# Check final cleaned keyword frequency
# ------------------------------------------------------------
final_counter = Counter()

for kws in df_biblio["clean_keywords"]:
    for kw in kws:
        final_counter[kw] += 1

final_keyword_freq = (
    pd.DataFrame(final_counter.items(), columns=["Keyword", "Frequency"])
    .sort_values("Frequency", ascending=False)
    .reset_index(drop=True)
)

final_keyword_freq.insert(0, "Rank", np.arange(1, len(final_keyword_freq) + 1))

display(final_keyword_freq.head(100))

final_keyword_freq.to_csv(
    TABLE_DIR / "final_clean_keyword_frequency_top_terms.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Final cleaned keyword coverage")
print("------------------------------")
print(f"Documents with at least one final keyword: {(df_biblio['n_clean_keywords'] > 0).sum():,}")
print(f"Documents without final keywords:          {(df_biblio['n_clean_keywords'] == 0).sum():,}")
print()


# ------------------------------------------------------------
# Define publication periods
# ------------------------------------------------------------
def assign_period(year):
    year = int(year)

    if 2010 <= year <= 2014:
        return "2010–2014"
    elif 2015 <= year <= 2019:
        return "2015–2019"
    elif 2020 <= year <= 2025:
        return "2020–2025"
    else:
        return np.nan


df_biblio["period"] = df_biblio["Year"].apply(assign_period)


# ------------------------------------------------------------
# Helper function for keyword frequency
# ------------------------------------------------------------
def keyword_frequency_from_dataframe(dataframe, keyword_col="clean_keywords"):
    counter = Counter()

    for kws in dataframe[keyword_col]:
        kws = ensure_keyword_list(kws)

        for kw in kws:
            counter[kw] += 1

    freq_df = (
        pd.DataFrame(counter.items(), columns=["Keyword", "Frequency"])
        .sort_values("Frequency", ascending=False)
        .reset_index(drop=True)
    )

    freq_df.insert(0, "Rank", np.arange(1, len(freq_df) + 1))

    return freq_df


# ------------------------------------------------------------
# Overall top 20
# ------------------------------------------------------------
overall_keyword_top20 = keyword_frequency_from_dataframe(df_biblio).head(20)

print("Overall top 20 keywords")
print("-----------------------")
display(overall_keyword_top20)


# ------------------------------------------------------------
# Period-specific top 20
# ------------------------------------------------------------
period_tables = {}

for period in ["2010–2014", "2015–2019", "2020–2025"]:
    temp = df_biblio[df_biblio["period"] == period].copy()
    period_tables[period] = keyword_frequency_from_dataframe(temp).head(20)

period_keyword_table = pd.DataFrame({"Rank": range(1, 21)})

for period in ["2010–2014", "2015–2019", "2020–2025"]:
    temp = period_tables[period].copy()

    period_keyword_table[period] = (
        temp["Keyword"].astype(str)
        + " ("
        + temp["Frequency"].astype(int).astype(str)
        + ")"
    )

print("Period-specific top 20 keywords")
print("--------------------------------")
display(period_keyword_table)


# ------------------------------------------------------------
# Manuscript-style combined table
# ------------------------------------------------------------
overall_for_table = overall_keyword_top20.copy()
overall_for_table["Overall"] = (
    overall_for_table["Keyword"].astype(str)
    + " ("
    + overall_for_table["Frequency"].astype(int).astype(str)
    + ")"
)

overall_compact = pd.DataFrame({
    "Rank": overall_for_table["Rank"],
    "Overall": overall_for_table["Overall"]
})

combined_keyword_table = overall_compact.merge(
    period_keyword_table,
    on="Rank",
    how="left"
)

print("Combined keyword table for manuscript")
print("-------------------------------------")
display(combined_keyword_table)


# ------------------------------------------------------------
# Save tables
# ------------------------------------------------------------
overall_keyword_top20.to_csv(
    TABLE_DIR / "table_overall_top20_keywords.csv",
    index=False,
    encoding="utf-8-sig"
)

period_keyword_table.to_csv(
    TABLE_DIR / "table_period_specific_top20_keywords.csv",
    index=False,
    encoding="utf-8-sig"
)

combined_keyword_table.to_csv(
    TABLE_DIR / "table_keyword_frequency_overall_and_by_period.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(TABLE_DIR / "final_clean_keyword_frequency_top_terms.csv")
print(TABLE_DIR / "table_overall_top20_keywords.csv")
print(TABLE_DIR / "table_period_specific_top20_keywords.csv")
print(TABLE_DIR / "table_keyword_frequency_overall_and_by_period.csv")

In [ ]:
# ============================================================
# CELL 22B — Final keyword refinement before co-occurrence analysis
# ============================================================

import ast
import re
import pandas as pd
import numpy as np
from collections import Counter

# ------------------------------------------------------------
# Helper: make sure keyword values are list-like
# ------------------------------------------------------------
def ensure_keyword_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    if isinstance(value, str):
        value = value.strip()

        if value == "":
            return []

        if value.startswith("[") and value.endswith("]"):
            try:
                parsed = ast.literal_eval(value)
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                pass

        return [v.strip() for v in re.split(r";|\|", value) if v.strip() != ""]

    return []

# ------------------------------------------------------------
# Additional final normalization
# ------------------------------------------------------------
final_keyword_normalization = {
    "bacteria (microorganisms": "bacteria",
    "bacteria (microorganism": "bacteria",
    "bacteria (microorganisms)": "bacteria",
    "bacteria (microorganism)": "bacteria",
    "bacterium": "bacteria",
    "bacteria": "bacteria",

    "dna extraction": "DNA extraction",
    "protein expression": "protein expression",
    "gene expression": "gene expression",

    "sensitivity and specificity": "sensitivity and specificity",
}

# ------------------------------------------------------------
# Additional final exclusion
# ------------------------------------------------------------
final_keyword_exclusion = {
    "molecular sequence data",
    "in vitro study",
    "in vitro studies",
    "dna extraction",
    "DNA extraction",
}

# ------------------------------------------------------------
# Refinement function
# ------------------------------------------------------------
def refine_clean_keyword(kw):
    if pd.isna(kw):
        return None

    kw = str(kw).strip()
    kw = re.sub(r"\s+", " ", kw)
    kw = kw.strip(" ,.;:")

    if kw == "":
        return None

    key = kw.casefold()
    key = re.sub(r"\s+", " ", key).strip()

    if key in {x.casefold() for x in final_keyword_exclusion}:
        return None

    if key in final_keyword_normalization:
        normalized = final_keyword_normalization[key]

        if normalized in final_keyword_exclusion:
            return None

        return normalized

    return kw

# ------------------------------------------------------------
# Apply final refinement
# ------------------------------------------------------------
def refine_keyword_list(keyword_list):
    keyword_list = ensure_keyword_list(keyword_list)

    refined = []

    for kw in keyword_list:
        new_kw = refine_clean_keyword(kw)

        if new_kw is not None:
            refined.append(new_kw)

    # Remove duplicates within each document
    seen = set()
    unique_refined = []

    for kw in refined:
        key = kw.casefold()

        if key not in seen:
            seen.add(key)
            unique_refined.append(kw)

    return unique_refined

df_biblio["clean_keywords"] = df_biblio["clean_keywords"].apply(refine_keyword_list)
df_biblio["n_clean_keywords"] = df_biblio["clean_keywords"].apply(len)

# ------------------------------------------------------------
# Check final keyword frequency
# ------------------------------------------------------------
final_counter = Counter()

for kws in df_biblio["clean_keywords"]:
    for kw in kws:
        final_counter[kw] += 1

final_keyword_freq = (
    pd.DataFrame(final_counter.items(), columns=["keyword", "frequency"])
    .sort_values("frequency", ascending=False)
    .reset_index(drop=True)
)

display(final_keyword_freq.head(50))

final_keyword_freq.to_csv(
    TABLE_DIR / "final_clean_keyword_frequency_top_terms.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Final cleaned keyword coverage")
print("------------------------------")
print(f"Documents with at least one final keyword: {(df_biblio['n_clean_keywords'] > 0).sum():,}")
print(f"Documents without final keywords:          {(df_biblio['n_clean_keywords'] == 0).sum():,}")
print()
print("Saved:")
print(TABLE_DIR / "final_clean_keyword_frequency_top_terms.csv")

In [ ]:
# ============================================================
# CELL 23 — Create keyword long table for co-occurrence analysis
# ============================================================

import pandas as pd
import numpy as np
import re
import ast

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------
def ensure_keyword_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    if isinstance(value, str):
        value = value.strip()

        if value == "":
            return []

        if value.startswith("[") and value.endswith("]"):
            try:
                parsed = ast.literal_eval(value)
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                pass

        return [v.strip() for v in re.split(r";|\|", value) if v.strip() != ""]

    return []

# ------------------------------------------------------------
# Make sure period column exists
# ------------------------------------------------------------
def assign_period(year):
    year = int(year)

    if 2010 <= year <= 2014:
        return "2010–2014"
    elif 2015 <= year <= 2019:
        return "2015–2019"
    elif 2020 <= year <= 2025:
        return "2020–2025"
    else:
        return np.nan

if "period" not in df_biblio.columns:
    df_biblio["period"] = df_biblio["Year"].apply(assign_period)

# ------------------------------------------------------------
# Select document identifier
# ------------------------------------------------------------
candidate_doc_id_cols = [
    "EID",
    "eid",
    "DOI",
    "doi",
    "Title",
    "title"
]

doc_id_col = None

for col in candidate_doc_id_cols:
    if col in df_biblio.columns:
        doc_id_col = col
        break

print("Document ID column:", doc_id_col)

# ------------------------------------------------------------
# Create keyword long table
# ------------------------------------------------------------
records = []

for idx, row in df_biblio.iterrows():
    if doc_id_col is not None and pd.notna(row[doc_id_col]) and str(row[doc_id_col]).strip() != "":
        document_id = str(row[doc_id_col])
    else:
        document_id = f"doc_{idx}"

    year = row["Year"]
    period = row["period"]

    keywords = ensure_keyword_list(row["clean_keywords"])

    for kw in keywords:
        records.append({
            "document_id": document_id,
            "Year": year,
            "period": period,
            "Keyword": kw
        })

keyword_long = pd.DataFrame(records)

# ------------------------------------------------------------
# Keyword document frequency
# ------------------------------------------------------------
keyword_document_frequency = (
    keyword_long
    .drop_duplicates(["document_id", "Keyword"])
    .groupby("Keyword")
    .size()
    .reset_index(name="document_frequency")
    .sort_values("document_frequency", ascending=False)
    .reset_index(drop=True)
)

keyword_document_frequency.insert(
    0,
    "Rank",
    np.arange(1, len(keyword_document_frequency) + 1)
)

display(keyword_document_frequency.head(50))

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
keyword_long.to_csv(
    TABLE_DIR / "keyword_long_table.csv",
    index=False,
    encoding="utf-8-sig"
)

keyword_document_frequency.to_csv(
    TABLE_DIR / "keyword_document_frequency.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Keyword long table")
print("------------------")
print(f"Rows:                 {len(keyword_long):,}")
print(f"Unique documents:     {keyword_long['document_id'].nunique():,}")
print(f"Unique keywords:      {keyword_long['Keyword'].nunique():,}")
print()
print("Saved:")
print(TABLE_DIR / "keyword_long_table.csv")
print(TABLE_DIR / "keyword_document_frequency.csv")

In [ ]:
# ============================================================
# CELL 24 — Create keyword co-occurrence pair table
# ============================================================

from itertools import combinations
from collections import Counter
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Network keyword settings
# ------------------------------------------------------------
top_n_network_keywords = 40

top_keywords = (
    keyword_document_frequency
    .head(top_n_network_keywords)["Keyword"]
    .tolist()
)

top_keyword_set = set(top_keywords)

keyword_rank = {
    kw: rank
    for rank, kw in enumerate(top_keywords, start=1)
}

keyword_freq_dict = (
    keyword_document_frequency
    .set_index("Keyword")["document_frequency"]
    .to_dict()
)

print("Top keywords used for network:")
print(top_keywords)

# ------------------------------------------------------------
# Count co-occurring keyword pairs within each document
# ------------------------------------------------------------
pair_counter = Counter()
document_counter = 0

for idx, row in df_biblio.iterrows():
    keywords = ensure_keyword_list(row["clean_keywords"])

    # Keep only top network keywords
    keywords = [
        kw for kw in keywords
        if kw in top_keyword_set
    ]

    # Remove duplicates and sort by frequency rank
    keywords = sorted(
        set(keywords),
        key=lambda x: keyword_rank.get(x, 9999)
    )

    if len(keywords) < 2:
        continue

    document_counter += 1

    for kw1, kw2 in combinations(keywords, 2):
        pair_counter[(kw1, kw2)] += 1

# ------------------------------------------------------------
# Convert to dataframe
# ------------------------------------------------------------
pair_records = []

for (kw1, kw2), cooc in pair_counter.items():
    freq1 = keyword_freq_dict.get(kw1, 0)
    freq2 = keyword_freq_dict.get(kw2, 0)

    association_strength = cooc / np.sqrt(freq1 * freq2) if freq1 > 0 and freq2 > 0 else np.nan

    pair_records.append({
        "Keyword_1": kw1,
        "Keyword_2": kw2,
        "cooccurrence": cooc,
        "frequency_1": freq1,
        "frequency_2": freq2,
        "association_strength": association_strength
    })

keyword_pair_table = (
    pd.DataFrame(pair_records)
    .sort_values(
        ["cooccurrence", "association_strength"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Select edges for network visualization
# ------------------------------------------------------------
min_edge_weight = 80
top_edges_for_network = 120

network_edge_table = (
    keyword_pair_table
    [keyword_pair_table["cooccurrence"] >= min_edge_weight]
    .head(top_edges_for_network)
    .copy()
    .reset_index(drop=True)
)

display(keyword_pair_table.head(50))
display(network_edge_table.head(50))

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
keyword_pair_table.to_csv(
    TABLE_DIR / "keyword_cooccurrence_pair_table.csv",
    index=False,
    encoding="utf-8-sig"
)

network_edge_table.to_csv(
    TABLE_DIR / "keyword_network_edges_for_figure4.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Co-occurrence analysis")
print("----------------------")
print(f"Documents with at least two top keywords: {document_counter:,}")
print(f"Total keyword pairs:                    {len(keyword_pair_table):,}")
print(f"Network edges used in Figure 4:         {len(network_edge_table):,}")
print(f"Minimum edge weight:                    {min_edge_weight:,}")
print(f"Top edges limit:                        {top_edges_for_network:,}")
print()
print("Saved:")
print(TABLE_DIR / "keyword_cooccurrence_pair_table.csv")
print(TABLE_DIR / "keyword_network_edges_for_figure4.csv")

In [ ]:
# ============================================================
# CELL 25 — Build keyword co-occurrence network and calculate centrality
# Revised: Hwang-compatible normalized closeness centrality
# ============================================================

import networkx as nx
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Build graph
# ------------------------------------------------------------
G = nx.Graph()

# Add edges and nodes
for _, row in network_edge_table.iterrows():
    kw1 = row["Keyword_1"]
    kw2 = row["Keyword_2"]
    weight = float(row["cooccurrence"])
    association_strength = float(row["association_strength"])

    for kw in [kw1, kw2]:
        if kw not in G:
            G.add_node(
                kw,
                frequency=int(keyword_freq_dict.get(kw, 0))
            )

    G.add_edge(
        kw1,
        kw2,
        weight=weight,
        association_strength=association_strength,
        distance=1 / weight if weight > 0 else 9999
    )

# ------------------------------------------------------------
# Remove isolates if any
# ------------------------------------------------------------
isolates = list(nx.isolates(G))

if len(isolates) > 0:
    G.remove_nodes_from(isolates)

# ------------------------------------------------------------
# Keep largest connected component if graph is disconnected
# ------------------------------------------------------------
if not nx.is_connected(G):
    components = list(nx.connected_components(G))
    largest_component = max(components, key=len)

    removed_nodes = sorted(list(set(G.nodes()) - set(largest_component)))

    print("Graph was disconnected.")
    print(f"Connected components: {len(components):,}")
    print(f"Nodes retained in largest component: {len(largest_component):,}")
    print(f"Nodes removed from smaller components: {len(removed_nodes):,}")
    print("Removed nodes:", removed_nodes)

    G = G.subgraph(largest_component).copy()

print("Network summary")
print("---------------")
print(f"Nodes:             {G.number_of_nodes():,}")
print(f"Edges:             {G.number_of_edges():,}")
print(f"Isolates removed:  {len(isolates):,}")

# ------------------------------------------------------------
# Centrality measures
# ------------------------------------------------------------

# Degree centrality
# Proportion of directly connected keyword nodes
degree_centrality = nx.degree_centrality(G)

# Weighted degree
# Sum of co-occurrence frequencies connected to each keyword
weighted_degree = {
    node: sum(
        edge_data["weight"]
        for _, _, edge_data in G.edges(node, data=True)
    )
    for node in G.nodes()
}

# ------------------------------------------------------------
# Hwang-compatible closeness centrality
# ------------------------------------------------------------
# NetworkX default closeness centrality is:
# Cc(v) = (N - 1) / sum d(v, u)
#
# This gives values in a more interpretable 0–1 range,
# but the maximum value does not have to be 1.
#
# The graph is treated as unweighted here because this is closer
# to conventional keyword-network centrality tables.
# ------------------------------------------------------------
closeness_centrality = nx.closeness_centrality(
    G,
    distance=None,
    wf_improved=True
)

# Betweenness centrality
# Bridging role between keyword groups
# Unweighted version is used for consistency with closeness centrality.
betweenness_centrality = nx.betweenness_centrality(
    G,
    weight=None,
    normalized=True
)

# Eigenvector centrality
# Connection to other influential keywords
# Unweighted version is used for consistency with topological centrality.
try:
    eigenvector_centrality = nx.eigenvector_centrality(
        G,
        weight=None,
        max_iter=2000,
        tol=1e-06
    )

except Exception as e:
    print("Eigenvector centrality did not converge.")
    print(e)

    eigenvector_centrality = nx.eigenvector_centrality_numpy(
        G,
        weight=None
    )

# ------------------------------------------------------------
# Community detection
# ------------------------------------------------------------
communities = list(
    nx.algorithms.community.greedy_modularity_communities(
        G,
        weight="weight"
    )
)

community_map = {}

for i, community in enumerate(communities, start=1):
    for node in community:
        community_map[node] = i

# ------------------------------------------------------------
# Centrality table
# ------------------------------------------------------------
centrality_records = []

for node in G.nodes():
    centrality_records.append({
        "Keyword": node,
        "Frequency": G.nodes[node]["frequency"],
        "Weighted_degree": weighted_degree[node],
        "Degree_centrality": degree_centrality[node],
        "Closeness_centrality": closeness_centrality[node],
        "Betweenness_centrality": betweenness_centrality[node],
        "Eigenvector_centrality": eigenvector_centrality[node],
        "Community": community_map.get(node, np.nan)
    })

keyword_centrality_table = (
    pd.DataFrame(centrality_records)
    .sort_values(
        ["Weighted_degree", "Frequency"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

keyword_centrality_table.insert(
    0,
    "Rank",
    np.arange(1, len(keyword_centrality_table) + 1)
)

display(keyword_centrality_table.head(30))

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
keyword_centrality_table.to_csv(
    TABLE_DIR / "keyword_network_centrality_table.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Communities")
print("-----------")

for i, community in enumerate(communities, start=1):
    community_keywords = sorted(list(community))
    print(f"Community {i}: {len(community_keywords)} keywords")
    print(", ".join(community_keywords))
    print()

print("Saved:")
print(TABLE_DIR / "keyword_network_centrality_table.csv")

print()
print("Centrality calculation note")
print("---------------------------")
print("Closeness centrality was calculated as normalized unweighted closeness centrality:")
print("Cc(v) = (N - 1) / sum d(v, u)")
print("Therefore, the maximum closeness value does not have to be 1.")

In [ ]:
# ============================================================
# CELL 26 — Figure 4. Keyword co-occurrence network
# Revised: label all nodes and remove legend
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import networkx as nx
from matplotlib import font_manager

# ------------------------------------------------------------
# Font check
# ------------------------------------------------------------
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
print("Arial installed:", "Arial" in available_fonts)

plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.unicode_minus"] = False

print("Current font.family setting:", plt.rcParams["font.family"])

# ------------------------------------------------------------
# Figure settings
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.6, 6.0))

layout_seed = 42

# 모든 노드에 라벨 표시
label_top_n = G.number_of_nodes()

# ------------------------------------------------------------
# Network layout
# k 값을 키우면 노드 간 거리가 조금 벌어짐
# ------------------------------------------------------------
pos = nx.spring_layout(
    G,
    seed=layout_seed,
    k=0.75,
    iterations=1500,
    weight="weight"
)

# ------------------------------------------------------------
# Edge width scaling
# ------------------------------------------------------------
edge_weights = np.array([
    G[u][v]["weight"]
    for u, v in G.edges()
])

edge_min = edge_weights.min()
edge_max = edge_weights.max()

edge_widths = [
    0.3 + 2.6 * ((G[u][v]["weight"] - edge_min) / (edge_max - edge_min))
    if edge_max > edge_min else 1.0
    for u, v in G.edges()
]

# ------------------------------------------------------------
# Node size scaling
# ------------------------------------------------------------
node_freqs = np.array([
    G.nodes[node]["frequency"]
    for node in G.nodes()
])

node_min = node_freqs.min()
node_max = node_freqs.max()

node_sizes = [
    280 + 1700 * ((G.nodes[node]["frequency"] - node_min) / (node_max - node_min))
    if node_max > node_min else 800
    for node in G.nodes()
]

# ------------------------------------------------------------
# Node colors by community
# ------------------------------------------------------------
community_palette = [
    "#E6B8CF",
    "#C6E0B4",
    "#B7D7E8",
    "#FCE4D6",
    "#D9D2E9",
    "#FFF2CC",
    "#D9EAD3",
    "#D0E0E3"
]

node_colors = []

for node in G.nodes():
    community_id = int(community_map.get(node, 1))
    node_colors.append(
        community_palette[(community_id - 1) % len(community_palette)]
    )

# ------------------------------------------------------------
# Draw edges
# ------------------------------------------------------------
nx.draw_networkx_edges(
    G,
    pos,
    ax=ax,
    width=edge_widths,
    edge_color="#A6A6A6",
    alpha=0.40
)

# ------------------------------------------------------------
# Draw nodes
# ------------------------------------------------------------
nx.draw_networkx_nodes(
    G,
    pos,
    ax=ax,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors="#4D4D4D",
    linewidths=0.6,
    alpha=0.95
)

# ------------------------------------------------------------
# Draw labels for all nodes
# ------------------------------------------------------------
label_keywords = (
    keyword_centrality_table
    .head(label_top_n)["Keyword"]
    .tolist()
)

labels = {
    node: node
    for node in G.nodes()
    if node in label_keywords
}

nx.draw_networkx_labels(
    G,
    pos,
    labels=labels,
    ax=ax,
    font_size=7.8,
    font_family="Arial",
    font_weight="normal",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.70,
        pad=0.20
    )
)

# ------------------------------------------------------------
# Final formatting
# ------------------------------------------------------------
ax.set_axis_off()
ax.margins(0.10)

plt.tight_layout()

fig_path = RESULTS_DIR / "figure_4_keyword_cooccurrence_network.png"
plt.savefig(fig_path, dpi=600, bbox_inches="tight")
plt.show()

print("Figure saved:")
print(fig_path)
print()
print("Figure 4 settings")
print("-----------------")
print(f"Nodes:              {G.number_of_nodes():,}")
print(f"Edges:              {G.number_of_edges():,}")
print(f"Labeled keywords:   {label_top_n}")
print(f"Layout seed:        {layout_seed}")

# ------------------------------------------------------------
# Check actual label font
# ------------------------------------------------------------
fig.canvas.draw()

label_fonts = sorted({
    text_obj.get_fontproperties().get_name()
    for text_obj in ax.texts
})

print("Actual network label font(s):", label_fonts)

In [ ]:
# ============================================================
# CELL 27 — Table 3. Central keywords in keyword co-occurrence network
# Revised: added closeness centrality
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Select top central keywords
# ------------------------------------------------------------
top_n_central_keywords = 20

table3_central_keywords = (
    keyword_centrality_table
    .sort_values(
        ["Weighted_degree", "Frequency"],
        ascending=[False, False]
    )
    .head(top_n_central_keywords)
    .copy()
    .reset_index(drop=True)
)

table3_central_keywords["Rank"] = np.arange(1, len(table3_central_keywords) + 1)

# ------------------------------------------------------------
# Format for manuscript table
# ------------------------------------------------------------
table3_central_keywords = table3_central_keywords[
    [
        "Rank",
        "Keyword",
        "Frequency",
        "Weighted_degree",
        "Degree_centrality",
        "Closeness_centrality",
        "Betweenness_centrality",
        "Eigenvector_centrality",
        "Community"
    ]
].copy()

table3_central_keywords = table3_central_keywords.rename(
    columns={
        "Weighted_degree": "Weighted degree",
        "Degree_centrality": "Degree centrality",
        "Closeness_centrality": "Closeness centrality",
        "Betweenness_centrality": "Betweenness centrality",
        "Eigenvector_centrality": "Eigenvector centrality"
    }
)

table3_central_keywords["Weighted degree"] = (
    table3_central_keywords["Weighted degree"]
    .round(0)
    .astype(int)
)

table3_central_keywords["Degree centrality"] = (
    table3_central_keywords["Degree centrality"]
    .round(4)
)

table3_central_keywords["Closeness centrality"] = (
    table3_central_keywords["Closeness centrality"]
    .round(4)
)

table3_central_keywords["Betweenness centrality"] = (
    table3_central_keywords["Betweenness centrality"]
    .round(4)
)

table3_central_keywords["Eigenvector centrality"] = (
    table3_central_keywords["Eigenvector centrality"]
    .round(4)
)

display(table3_central_keywords)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
table3_central_keywords.to_csv(
    TABLE_DIR / "table3_central_keywords_cooccurrence_network.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(TABLE_DIR / "table3_central_keywords_cooccurrence_network.csv")

In [ ]:
# ============================================================
# CELL 27B — Centrality-specific ranking tables
# Keyword + score by each centrality metric
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
top_n_per_metric = 20

centrality_metrics = {
    "Weighted degree": "Weighted_degree",
    "Degree centrality": "Degree_centrality",
    "Closeness centrality": "Closeness_centrality",
    "Betweenness centrality": "Betweenness_centrality",
    "Eigenvector centrality": "Eigenvector_centrality"
}

# ------------------------------------------------------------
# Check required columns
# ------------------------------------------------------------
required_cols = ["Keyword", "Frequency"] + list(centrality_metrics.values())

missing_cols = [
    col for col in required_cols
    if col not in keyword_centrality_table.columns
]

if len(missing_cols) > 0:
    raise ValueError(f"Missing columns in keyword_centrality_table: {missing_cols}")

# ------------------------------------------------------------
# Create long-format ranking table
# ------------------------------------------------------------
ranking_tables = []

for metric_name, metric_col in centrality_metrics.items():
    temp = (
        keyword_centrality_table
        .sort_values(metric_col, ascending=False)
        .head(top_n_per_metric)
        .copy()
        .reset_index(drop=True)
    )

    temp["Rank"] = np.arange(1, len(temp) + 1)
    temp["Metric"] = metric_name
    temp["Score"] = temp[metric_col]

    temp = temp[
        [
            "Metric",
            "Rank",
            "Keyword",
            "Score",
            "Frequency"
        ]
    ]

    ranking_tables.append(temp)

centrality_rank_long = pd.concat(
    ranking_tables,
    axis=0,
    ignore_index=True
)

# ------------------------------------------------------------
# Rounding for display
# ------------------------------------------------------------
centrality_rank_long_display = centrality_rank_long.copy()

centrality_rank_long_display["Score"] = centrality_rank_long_display.apply(
    lambda row: round(row["Score"], 0)
    if row["Metric"] == "Weighted degree"
    else round(row["Score"], 4),
    axis=1
)

display(centrality_rank_long_display)

# ------------------------------------------------------------
# Create wide-format table:
# Rank | Weighted degree | Degree centrality | ...
# Each cell = keyword (score)
# ------------------------------------------------------------
wide_records = []

for rank in range(1, top_n_per_metric + 1):
    row_record = {"Rank": rank}

    for metric_name, metric_col in centrality_metrics.items():
        temp = centrality_rank_long[
            (centrality_rank_long["Metric"] == metric_name)
            & (centrality_rank_long["Rank"] == rank)
        ]

        if len(temp) == 0:
            row_record[metric_name] = ""
            continue

        keyword = temp.iloc[0]["Keyword"]
        score = temp.iloc[0]["Score"]

        if metric_name == "Weighted degree":
            score_text = f"{score:,.0f}"
        else:
            score_text = f"{score:.4f}"

        row_record[metric_name] = f"{keyword} ({score_text})"

    wide_records.append(row_record)

centrality_rank_wide = pd.DataFrame(wide_records)

display(centrality_rank_wide)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
centrality_rank_long_display.to_csv(
    TABLE_DIR / "centrality_rankings_by_metric_long.csv",
    index=False,
    encoding="utf-8-sig"
)

centrality_rank_wide.to_csv(
    TABLE_DIR / "centrality_rankings_by_metric_wide.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(TABLE_DIR / "centrality_rankings_by_metric_long.csv")
print(TABLE_DIR / "centrality_rankings_by_metric_wide.csv")

In [ ]:
# ============================================================
# CELL 28 — Prepare BERTopic input corpus
# ============================================================

import pandas as pd
import numpy as np
import re
from pathlib import Path

# ------------------------------------------------------------
# Load BERTopic dataset if not already in memory
# ------------------------------------------------------------
if "df_bertopic" not in globals():
    bertopic_data_path = PROCESSED_DIR / "veterinary_science_bertopic_dataset_2010_2025.csv"
    
    if bertopic_data_path.exists():
        df_bertopic = pd.read_csv(bertopic_data_path, encoding="utf-8-sig", low_memory=False)
        print("Loaded df_bertopic from:")
        print(bertopic_data_path)
    else:
        raise FileNotFoundError(
            "df_bertopic was not found in memory and the processed BERTopic dataset file does not exist."
        )

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def find_column(dataframe, candidates):
    for col in candidates:
        if col in dataframe.columns:
            return col
    return None

def clean_text_for_bertopic(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    
    return text

# ------------------------------------------------------------
# Identify key columns
# ------------------------------------------------------------
title_col = find_column(df_bertopic, ["Title", "title", "Document Title"])
abstract_col = find_column(df_bertopic, ["Abstract", "abstract", "Abstracts"])
year_col = find_column(df_bertopic, ["Year", "year", "Publication Year"])
journal_col = find_column(df_bertopic, ["source_title_clean", "Source title", "Source Title", "journal"])
doi_col = find_column(df_bertopic, ["DOI", "doi"])
eid_col = find_column(df_bertopic, ["EID", "eid"])

print("Detected columns")
print("----------------")
print("Title column:    ", title_col)
print("Abstract column: ", abstract_col)
print("Year column:     ", year_col)
print("Journal column:  ", journal_col)
print("DOI column:      ", doi_col)
print("EID column:      ", eid_col)

if title_col is None:
    raise ValueError("Title column was not found.")

if abstract_col is None:
    raise ValueError("Abstract column was not found.")

if year_col is None:
    raise ValueError("Year column was not found.")

# ------------------------------------------------------------
# Prepare text columns
# ------------------------------------------------------------
df_topic = df_bertopic.copy()

df_topic["title_text_for_topic"] = df_topic[title_col].apply(clean_text_for_bertopic)
df_topic["abstract_text_for_topic"] = df_topic[abstract_col].apply(clean_text_for_bertopic)

df_topic["text_title_only"] = df_topic["title_text_for_topic"]
df_topic["text_abstract_only"] = df_topic["abstract_text_for_topic"]

df_topic["text_title_abstract"] = (
    df_topic["title_text_for_topic"]
    + ". "
    + df_topic["abstract_text_for_topic"]
).str.strip()

# ------------------------------------------------------------
# Main corpus setting
# ------------------------------------------------------------
main_text_col = "text_title_abstract"

df_topic["bertopic_text"] = df_topic[main_text_col].apply(clean_text_for_bertopic)
df_topic["bertopic_word_count"] = df_topic["bertopic_text"].apply(lambda x: len(str(x).split()))

# ------------------------------------------------------------
# Filter very short records
# ------------------------------------------------------------
min_word_count = 25

df_topic = (
    df_topic[
        df_topic["bertopic_text"].notna()
        & (df_topic["bertopic_text"].str.strip() != "")
        & (df_topic["bertopic_word_count"] >= min_word_count)
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Create document ID
# ------------------------------------------------------------
def make_document_id(row, idx):
    if eid_col is not None and pd.notna(row[eid_col]) and str(row[eid_col]).strip() != "":
        return str(row[eid_col])
    
    if doi_col is not None and pd.notna(row[doi_col]) and str(row[doi_col]).strip() != "":
        return str(row[doi_col])
    
    return f"doc_{idx}"

df_topic["document_id"] = [
    make_document_id(row, idx)
    for idx, row in df_topic.iterrows()
]

# ------------------------------------------------------------
# Metadata for later analysis
# ------------------------------------------------------------
topic_docs = df_topic["bertopic_text"].tolist()
topic_doc_ids = df_topic["document_id"].tolist()
topic_years = df_topic[year_col].astype(int).tolist()

topic_metadata = pd.DataFrame({
    "document_id": df_topic["document_id"],
    "Year": df_topic[year_col].astype(int),
    "bertopic_word_count": df_topic["bertopic_word_count"]
})

if journal_col is not None:
    topic_metadata["Journal"] = df_topic[journal_col].astype(str)

if doi_col is not None:
    topic_metadata["DOI"] = df_topic[doi_col]

if title_col is not None:
    topic_metadata["Title"] = df_topic[title_col]

# ------------------------------------------------------------
# Save input metadata
# ------------------------------------------------------------
topic_metadata.to_csv(
    PROCESSED_DIR / "bertopic_input_metadata.csv",
    index=False,
    encoding="utf-8-sig"
)

print()
print("BERTopic corpus summary")
print("-----------------------")
print(f"Documents used for BERTopic: {len(topic_docs):,}")
print(f"Minimum word count:          {min_word_count}")
print(f"Text column used:            {main_text_col}")
print(f"Year range:                  {min(topic_years)}–{max(topic_years)}")
print()
print("Saved:")
print(PROCESSED_DIR / "bertopic_input_metadata.csv")

display(topic_metadata.head())

In [ ]:
# ============================================================
# CELL 29 — Generate or load sentence embeddings
# ============================================================

import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# Import sentence-transformers
# ------------------------------------------------------------
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    raise ImportError(
        "sentence-transformers is not installed. "
        "Install it using: pip install sentence-transformers"
    )

# ------------------------------------------------------------
# Embedding settings
# ------------------------------------------------------------
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_batch_size = 64

EMBEDDING_DIR = RESULTS_DIR / "embeddings"
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

safe_model_name = (
    embedding_model_name
    .replace("/", "_")
    .replace("-", "_")
)

embedding_path = EMBEDDING_DIR / f"embeddings_{safe_model_name}_{len(topic_docs)}docs.npy"

print("Embedding model:")
print(embedding_model_name)
print()
print("Embedding file:")
print(embedding_path)

# ------------------------------------------------------------
# Load existing embeddings if available
# ------------------------------------------------------------
if embedding_path.exists():
    embeddings = np.load(embedding_path)
    print()
    print("Existing embeddings loaded.")
    print("Embedding shape:", embeddings.shape)

else:
    print()
    print("Generating embeddings. This may take some time...")
    
    embedding_model = SentenceTransformer(embedding_model_name)
    
    embeddings = embedding_model.encode(
        topic_docs,
        batch_size=embedding_batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    embeddings = embeddings.astype("float32")
    np.save(embedding_path, embeddings)
    
    print()
    print("Embeddings generated and saved.")
    print("Embedding shape:", embeddings.shape)

# ------------------------------------------------------------
# Load model object for BERTopic
# ------------------------------------------------------------
embedding_model = SentenceTransformer(embedding_model_name)

print()
print("Final embedding shape:")
print(embeddings.shape)

In [ ]:
# ============================================================
# CELL 30 — Fit baseline BERTopic model
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# Import BERTopic-related packages
# ------------------------------------------------------------
try:
    from bertopic import BERTopic
except ImportError:
    raise ImportError(
        "BERTopic is not installed. "
        "Install it using: pip install bertopic"
    )

try:
    from umap import UMAP
except ImportError:
    raise ImportError(
        "umap-learn is not installed. "
        "Install it using: pip install umap-learn"
    )

try:
    import hdbscan
except ImportError:
    raise ImportError(
        "hdbscan is not installed. "
        "Install it using: pip install hdbscan"
    )

from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from bertopic.vectorizers import ClassTfidfTransformer

# ------------------------------------------------------------
# BERTopic output directories
# ------------------------------------------------------------
BERTOPIC_DIR = RESULTS_DIR / "bertopic"
BERTOPIC_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Custom stop words
# Remove overly broad corpus-level words
# ------------------------------------------------------------
custom_stop_words = set(ENGLISH_STOP_WORDS)

custom_stop_words.update([
    "study",
    "studies",
    "result",
    "results",
    "method",
    "methods",
    "conclusion",
    "conclusions",
    "background",
    "objective",
    "objectives",
    "aim",
    "aims",
    "using",
    "used",
    "based",
    "analysis",
    "data",
    "group",
    "groups",
    "significant",
    "significantly",
    "increase",
    "increased",
    "decrease",
    "decreased",
    "effect",
    "effects",
    "associated",
    "association",
    "evaluated",
    "investigated",
    "observed",
    "including",
    "respectively",
    "compared",
    "control",
    "treatment",
    "sample",
    "samples",
    "animal",
    "animals",
    "veterinary",
    "medicine",
    "science",
    "clinical",
    "case",
    "cases",
    "disease",
    "diseases"
])

custom_stop_words = sorted(list(custom_stop_words))

# ------------------------------------------------------------
# Model parameters
# ------------------------------------------------------------
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=120,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words=custom_stop_words,
    ngram_range=(1, 2),
    min_df=20,
    max_df=0.65
)

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    language="english",
    top_n_words=10,
    calculate_probabilities=False,
    verbose=True
)

# ------------------------------------------------------------
# Fit model
# ------------------------------------------------------------
topics, probabilities = topic_model.fit_transform(
    topic_docs,
    embeddings
)

topics = np.array(topics)

# ------------------------------------------------------------
# Topic information
# ------------------------------------------------------------
topic_info = topic_model.get_topic_info()

n_documents = len(topics)
n_outliers = int(np.sum(topics == -1))
outlier_ratio = n_outliers / n_documents
n_raw_topics = topic_info[topic_info["Topic"] != -1]["Topic"].nunique()

print()
print("BERTopic baseline summary")
print("-------------------------")
print(f"Documents:       {n_documents:,}")
print(f"Raw topics:      {n_raw_topics:,}")
print(f"Outliers:        {n_outliers:,}")
print(f"Outlier ratio:   {outlier_ratio:.4f}")

display(topic_info.head(30))

# ------------------------------------------------------------
# Save topic assignments
# ------------------------------------------------------------
df_bertopic_results = topic_metadata.copy()
df_bertopic_results["Topic"] = topics

df_bertopic_results.to_csv(
    BERTOPIC_DIR / "bertopic_document_topic_assignments_baseline.csv",
    index=False,
    encoding="utf-8-sig"
)

topic_info.to_csv(
    BERTOPIC_DIR / "bertopic_topic_info_baseline.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Save model
# ------------------------------------------------------------
model_save_path = BERTOPIC_DIR / "bertopic_baseline_model"

topic_model.save(
    model_save_path,
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=False
)

print()
print("Saved:")
print(BERTOPIC_DIR / "bertopic_document_topic_assignments_baseline.csv")
print(BERTOPIC_DIR / "bertopic_topic_info_baseline.csv")
print(model_save_path)

In [ ]:
# ============================================================
# CELL 31 — Evaluate BERTopic baseline model
# Coherence, diversity, outlier ratio, and topic-size balance
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
top_n_words_eval = 10
coherence_sample_size = None
random_state = 42

# ------------------------------------------------------------
# Valid topics
# ------------------------------------------------------------
valid_topic_ids = sorted([
    topic_id
    for topic_id in topic_info["Topic"].tolist()
    if topic_id != -1
])

valid_topic_sizes = (
    topic_info[topic_info["Topic"] != -1]
    .set_index("Topic")["Count"]
)

# ------------------------------------------------------------
# Extract topic words
# ------------------------------------------------------------
topic_words = []

for topic_id in valid_topic_ids:
    words_scores = topic_model.get_topic(topic_id)
    
    if words_scores is None:
        continue
    
    words = [
        word
        for word, score in words_scores[:top_n_words_eval]
    ]
    
    if len(words) > 0:
        topic_words.append(words)

# ------------------------------------------------------------
# Topic diversity
# ------------------------------------------------------------
all_topic_words = [
    word
    for words in topic_words
    for word in words
]

topic_diversity = (
    len(set(all_topic_words)) / len(all_topic_words)
    if len(all_topic_words) > 0 else np.nan
)

# ------------------------------------------------------------
# Coherence calculation
# ------------------------------------------------------------
coherence_cv = np.nan

try:
    from gensim.corpora import Dictionary
    from gensim.models import CoherenceModel
    
    analyzer = topic_model.vectorizer_model.build_analyzer()
    
    if coherence_sample_size is None:
        docs_for_coherence = topic_docs
    else:
        rng = np.random.default_rng(random_state)
        sample_idx = rng.choice(
            len(topic_docs),
            size=min(coherence_sample_size, len(topic_docs)),
            replace=False
        )
        docs_for_coherence = [topic_docs[i] for i in sample_idx]
    
    tokenized_docs = [
        analyzer(doc)
        for doc in docs_for_coherence
    ]
    
    tokenized_docs = [
        tokens
        for tokens in tokenized_docs
        if len(tokens) > 0
    ]
    
    dictionary = Dictionary(tokenized_docs)
    
    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence="c_v"
    )
    
    coherence_cv = coherence_model.get_coherence()

except Exception as e:
    print("Coherence calculation failed.")
    print("Reason:")
    print(e)
    print()
    print("The model evaluation table will be created without coherence.")

# ------------------------------------------------------------
# Topic-size statistics
# ------------------------------------------------------------
n_documents = len(topics)
n_outliers = int(np.sum(topics == -1))
outlier_ratio = n_outliers / n_documents

n_valid_topics = len(valid_topic_ids)
n_valid_documents = int(np.sum(topics != -1))

min_topic_size = int(valid_topic_sizes.min()) if len(valid_topic_sizes) > 0 else np.nan
max_topic_size = int(valid_topic_sizes.max()) if len(valid_topic_sizes) > 0 else np.nan
median_topic_size = float(valid_topic_sizes.median()) if len(valid_topic_sizes) > 0 else np.nan

topic_size_cv = (
    float(valid_topic_sizes.std() / valid_topic_sizes.mean())
    if len(valid_topic_sizes) > 1 else np.nan
)

# ------------------------------------------------------------
# Evaluation table
# ------------------------------------------------------------
bertopic_baseline_evaluation = pd.DataFrame([{
    "Model": "baseline_title_abstract",
    "Embedding_model": embedding_model_name,
    "Text_input": main_text_col,
    "Documents": n_documents,
    "Valid_documents": n_valid_documents,
    "Outliers": n_outliers,
    "Outlier_ratio": round(outlier_ratio, 4),
    "Raw_topics": n_valid_topics,
    "Coherence_c_v": round(coherence_cv, 4) if pd.notna(coherence_cv) else np.nan,
    "Topic_diversity": round(topic_diversity, 4),
    "Min_topic_size": min_topic_size,
    "Median_topic_size": round(median_topic_size, 1) if pd.notna(median_topic_size) else np.nan,
    "Max_topic_size": max_topic_size,
    "Topic_size_CV": round(topic_size_cv, 4) if pd.notna(topic_size_cv) else np.nan,
    "UMAP_n_neighbors": 15,
    "UMAP_n_components": 5,
    "UMAP_min_dist": 0.0,
    "HDBSCAN_min_cluster_size": 120,
    "HDBSCAN_min_samples": 10,
    "Vectorizer_min_df": 20,
    "Vectorizer_max_df": 0.65,
    "Vectorizer_ngram_range": "1–2"
}])

display(bertopic_baseline_evaluation)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
bertopic_baseline_evaluation.to_csv(
    BERTOPIC_DIR / "bertopic_baseline_evaluation.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(BERTOPIC_DIR / "bertopic_baseline_evaluation.csv")

In [ ]:
# ============================================================
# CELL 32 — BERTopic parameter comparison
# Revised: safer vectorizer min_df/max_df settings
# ============================================================

import pandas as pd
import numpy as np
import math
from pathlib import Path

from bertopic import BERTopic
from umap import UMAP
import hdbscan

from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from bertopic.vectorizers import ClassTfidfTransformer

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
PARAM_DIR = BERTOPIC_DIR / "parameter_comparison"
PARAM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Custom stop words
# ------------------------------------------------------------
custom_stop_words = set(ENGLISH_STOP_WORDS)

custom_stop_words.update([
    "study",
    "studies",
    "result",
    "results",
    "method",
    "methods",
    "conclusion",
    "conclusions",
    "background",
    "objective",
    "objectives",
    "aim",
    "aims",
    "using",
    "used",
    "based",
    "analysis",
    "data",
    "group",
    "groups",
    "significant",
    "significantly",
    "increase",
    "increased",
    "decrease",
    "decreased",
    "effect",
    "effects",
    "associated",
    "association",
    "evaluated",
    "investigated",
    "observed",
    "including",
    "respectively",
    "compared",
    "control",
    "treatment",
    "sample",
    "samples",
    "animal",
    "animals",
    "veterinary",
    "medicine",
    "science",
    "clinical",
    "case",
    "cases",
    "disease",
    "diseases"
])

custom_stop_words = sorted(list(custom_stop_words))

# ------------------------------------------------------------
# Parameter grid
# Vectorizer_min_df was reduced because BERTopic applies CountVectorizer
# to topic-level aggregated documents, not to all original documents.
# ------------------------------------------------------------
parameter_grid = [
    {
        "Model": "M1_baseline",
        "UMAP_n_neighbors": 15,
        "UMAP_n_components": 5,
        "UMAP_min_dist": 0.0,
        "HDBSCAN_min_cluster_size": 120,
        "HDBSCAN_min_samples": 10,
        "Vectorizer_min_df": 5,
        "Vectorizer_max_df": 0.95
    },
    {
        "Model": "M2_larger_clusters",
        "UMAP_n_neighbors": 15,
        "UMAP_n_components": 5,
        "UMAP_min_dist": 0.0,
        "HDBSCAN_min_cluster_size": 200,
        "HDBSCAN_min_samples": 10,
        "Vectorizer_min_df": 5,
        "Vectorizer_max_df": 0.95
    },
    {
        "Model": "M3_more_conservative",
        "UMAP_n_neighbors": 15,
        "UMAP_n_components": 5,
        "UMAP_min_dist": 0.0,
        "HDBSCAN_min_cluster_size": 300,
        "HDBSCAN_min_samples": 10,
        "Vectorizer_min_df": 5,
        "Vectorizer_max_df": 0.95
    },
    {
        "Model": "M4_lower_outlier",
        "UMAP_n_neighbors": 30,
        "UMAP_n_components": 5,
        "UMAP_min_dist": 0.0,
        "HDBSCAN_min_cluster_size": 200,
        "HDBSCAN_min_samples": 5,
        "Vectorizer_min_df": 5,
        "Vectorizer_max_df": 0.95
    },
    {
        "Model": "M5_stable_broader",
        "UMAP_n_neighbors": 30,
        "UMAP_n_components": 5,
        "UMAP_min_dist": 0.0,
        "HDBSCAN_min_cluster_size": 300,
        "HDBSCAN_min_samples": 10,
        "Vectorizer_min_df": 5,
        "Vectorizer_max_df": 0.95
    },
    {
        "Model": "M6_high_min_df",
        "UMAP_n_neighbors": 30,
        "UMAP_n_components": 5,
        "UMAP_min_dist": 0.0,
        "HDBSCAN_min_cluster_size": 300,
        "HDBSCAN_min_samples": 10,
        "Vectorizer_min_df": 8,
        "Vectorizer_max_df": 0.95
    }
]

# ------------------------------------------------------------
# Evaluation helper
# ------------------------------------------------------------
def calculate_topic_diversity(model, valid_topic_ids, top_n_words=10):
    topic_words = []

    for topic_id in valid_topic_ids:
        words_scores = model.get_topic(topic_id)

        if words_scores is None:
            continue

        words = [
            word
            for word, score in words_scores[:top_n_words]
        ]

        if len(words) > 0:
            topic_words.append(words)

    all_words = [
        word
        for words in topic_words
        for word in words
    ]

    if len(all_words) == 0:
        return np.nan

    return len(set(all_words)) / len(all_words)

# ------------------------------------------------------------
# Fit and evaluate models
# ------------------------------------------------------------
comparison_records = []
model_objects = {}

for params in parameter_grid:
    model_name = params["Model"]

    print()
    print("=" * 80)
    print(f"Fitting {model_name}")
    print("=" * 80)

    try:
        umap_model = UMAP(
            n_neighbors=params["UMAP_n_neighbors"],
            n_components=params["UMAP_n_components"],
            min_dist=params["UMAP_min_dist"],
            metric="cosine",
            random_state=42
        )

        hdbscan_model = hdbscan.HDBSCAN(
            min_cluster_size=params["HDBSCAN_min_cluster_size"],
            min_samples=params["HDBSCAN_min_samples"],
            metric="euclidean",
            cluster_selection_method="eom",
            prediction_data=True
        )

        vectorizer_model = CountVectorizer(
            stop_words=custom_stop_words,
            ngram_range=(1, 2),
            min_df=params["Vectorizer_min_df"],
            max_df=params["Vectorizer_max_df"]
        )

        ctfidf_model = ClassTfidfTransformer(
            reduce_frequent_words=True
        )

        temp_topic_model = BERTopic(
            embedding_model=embedding_model,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            vectorizer_model=vectorizer_model,
            ctfidf_model=ctfidf_model,
            language="english",
            top_n_words=10,
            calculate_probabilities=False,
            verbose=True
        )

        temp_topics, _ = temp_topic_model.fit_transform(
            topic_docs,
            embeddings
        )

        temp_topics = np.array(temp_topics)
        temp_topic_info = temp_topic_model.get_topic_info()

        n_documents = len(temp_topics)
        n_outliers = int(np.sum(temp_topics == -1))
        n_valid_documents = int(np.sum(temp_topics != -1))
        outlier_ratio = n_outliers / n_documents

        valid_topic_info = temp_topic_info[temp_topic_info["Topic"] != -1].copy()
        valid_topic_ids = sorted(valid_topic_info["Topic"].tolist())

        n_valid_topics = len(valid_topic_ids)

        topic_sizes = valid_topic_info["Count"]

        min_topic_size = int(topic_sizes.min()) if len(topic_sizes) > 0 else np.nan
        median_topic_size = float(topic_sizes.median()) if len(topic_sizes) > 0 else np.nan
        max_topic_size = int(topic_sizes.max()) if len(topic_sizes) > 0 else np.nan

        topic_size_cv = (
            float(topic_sizes.std() / topic_sizes.mean())
            if len(topic_sizes) > 1 else np.nan
        )

        topic_diversity = calculate_topic_diversity(
            temp_topic_model,
            valid_topic_ids,
            top_n_words=10
        )

        comparison_records.append({
            "Model": model_name,
            "Status": "success",
            "Documents": n_documents,
            "Valid_documents": n_valid_documents,
            "Outliers": n_outliers,
            "Outlier_ratio": round(outlier_ratio, 4),
            "Raw_topics": n_valid_topics,
            "Topic_diversity": round(topic_diversity, 4),
            "Min_topic_size": min_topic_size,
            "Median_topic_size": round(median_topic_size, 1) if pd.notna(median_topic_size) else np.nan,
            "Max_topic_size": max_topic_size,
            "Topic_size_CV": round(topic_size_cv, 4) if pd.notna(topic_size_cv) else np.nan,
            "UMAP_n_neighbors": params["UMAP_n_neighbors"],
            "UMAP_n_components": params["UMAP_n_components"],
            "UMAP_min_dist": params["UMAP_min_dist"],
            "HDBSCAN_min_cluster_size": params["HDBSCAN_min_cluster_size"],
            "HDBSCAN_min_samples": params["HDBSCAN_min_samples"],
            "Vectorizer_min_df": params["Vectorizer_min_df"],
            "Vectorizer_max_df": params["Vectorizer_max_df"],
            "Error_message": ""
        })

        # Save topic info and assignments for each model
        temp_result_df = topic_metadata.copy()
        temp_result_df["Topic"] = temp_topics

        temp_result_df.to_csv(
            PARAM_DIR / f"{model_name}_document_topic_assignments.csv",
            index=False,
            encoding="utf-8-sig"
        )

        temp_topic_info.to_csv(
            PARAM_DIR / f"{model_name}_topic_info.csv",
            index=False,
            encoding="utf-8-sig"
        )

        model_objects[model_name] = {
            "model": temp_topic_model,
            "topics": temp_topics,
            "topic_info": temp_topic_info
        }

        print()
        print(f"{model_name} summary")
        print("--------------------")
        print(f"Valid topics:   {n_valid_topics:,}")
        print(f"Outliers:       {n_outliers:,}")
        print(f"Outlier ratio:  {outlier_ratio:.4f}")
        print(f"Diversity:      {topic_diversity:.4f}")
        print(f"Topic size CV:  {topic_size_cv:.4f}")

    except Exception as e:
        print()
        print(f"{model_name} failed.")
        print("Reason:")
        print(e)

        comparison_records.append({
            "Model": model_name,
            "Status": "failed",
            "Documents": len(topic_docs),
            "Valid_documents": np.nan,
            "Outliers": np.nan,
            "Outlier_ratio": np.nan,
            "Raw_topics": np.nan,
            "Topic_diversity": np.nan,
            "Min_topic_size": np.nan,
            "Median_topic_size": np.nan,
            "Max_topic_size": np.nan,
            "Topic_size_CV": np.nan,
            "UMAP_n_neighbors": params["UMAP_n_neighbors"],
            "UMAP_n_components": params["UMAP_n_components"],
            "UMAP_min_dist": params["UMAP_min_dist"],
            "HDBSCAN_min_cluster_size": params["HDBSCAN_min_cluster_size"],
            "HDBSCAN_min_samples": params["HDBSCAN_min_samples"],
            "Vectorizer_min_df": params["Vectorizer_min_df"],
            "Vectorizer_max_df": params["Vectorizer_max_df"],
            "Error_message": str(e)
        })

# ------------------------------------------------------------
# Comparison table
# ------------------------------------------------------------
bertopic_parameter_comparison = pd.DataFrame(comparison_records)

# Put successful models first
bertopic_parameter_comparison["Status_order"] = bertopic_parameter_comparison["Status"].map({
    "success": 0,
    "failed": 1
})

bertopic_parameter_comparison = (
    bertopic_parameter_comparison
    .sort_values(
        ["Status_order", "Outlier_ratio", "Raw_topics", "Topic_size_CV"],
        ascending=[True, True, True, True]
    )
    .drop(columns=["Status_order"])
    .reset_index(drop=True)
)

display(bertopic_parameter_comparison)

bertopic_parameter_comparison.to_csv(
    BERTOPIC_DIR / "bertopic_parameter_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print()
print("Saved:")
print(BERTOPIC_DIR / "bertopic_parameter_comparison.csv")
print(PARAM_DIR)

In [ ]:
# ============================================================
# CELL 33 — Select final BERTopic model and create final topic table
# Final model: M3_more_conservative
# ============================================================

import pandas as pd
import numpy as np
import re
from pathlib import Path

# ------------------------------------------------------------
# Select final model
# ------------------------------------------------------------
final_model_name = "M3_more_conservative"

if "model_objects" not in globals():
    raise ValueError(
        "model_objects was not found in memory. "
        "Please re-run Cell 32 or load/refit the selected final model."
    )

if final_model_name not in model_objects:
    raise ValueError(
        f"{final_model_name} was not found in model_objects. "
        f"Available models: {list(model_objects.keys())}"
    )

final_topic_model = model_objects[final_model_name]["model"]
final_topics = np.array(model_objects[final_model_name]["topics"])
final_topic_info = model_objects[final_model_name]["topic_info"].copy()

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
FINAL_TOPIC_DIR = BERTOPIC_DIR / "final_model"
FINAL_TOPIC_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Basic summary
# ------------------------------------------------------------
n_documents = len(final_topics)
n_outliers = int(np.sum(final_topics == -1))
n_valid_documents = int(np.sum(final_topics != -1))
outlier_ratio = n_outliers / n_documents

valid_topic_info = final_topic_info[final_topic_info["Topic"] != -1].copy()
n_valid_topics = valid_topic_info["Topic"].nunique()

print("Final BERTopic model summary")
print("----------------------------")
print(f"Final model:       {final_model_name}")
print(f"Documents:         {n_documents:,}")
print(f"Valid documents:   {n_valid_documents:,}")
print(f"Outliers:          {n_outliers:,}")
print(f"Outlier ratio:     {outlier_ratio:.4f}")
print(f"Valid topics:      {n_valid_topics:,}")

# ------------------------------------------------------------
# Save final document-topic assignments
# ------------------------------------------------------------
final_document_topic_assignments = topic_metadata.copy()
final_document_topic_assignments["Topic"] = final_topics

final_document_topic_assignments.to_csv(
    FINAL_TOPIC_DIR / "final_document_topic_assignments.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def clean_topic_name(name):
    name = str(name)
    name = re.sub(r"^\-?\d+_", "", name)
    name = name.replace("_", ", ")
    name = re.sub(r"\s+", " ", name)
    return name.strip()

def get_topic_words_as_string(topic_model, topic_id, top_n=10):
    topic_words = topic_model.get_topic(topic_id)
    
    if topic_words is None:
        return ""
    
    words = [
        word
        for word, score in topic_words[:top_n]
    ]
    
    return ", ".join(words)

def get_representative_titles(topic_id, n=3):
    temp = final_document_topic_assignments[
        final_document_topic_assignments["Topic"] == topic_id
    ].copy()
    
    if "Title" not in temp.columns:
        return ""
    
    titles = (
        temp["Title"]
        .dropna()
        .astype(str)
        .head(n)
        .tolist()
    )
    
    return " | ".join(titles)

# ------------------------------------------------------------
# Create final topic table
# ------------------------------------------------------------
final_topic_table = valid_topic_info.copy()

final_topic_table["Topic_label_raw"] = final_topic_table["Name"].apply(clean_topic_name)

final_topic_table["Top_words"] = final_topic_table["Topic"].apply(
    lambda x: get_topic_words_as_string(final_topic_model, x, top_n=10)
)

final_topic_table["Representative_titles"] = final_topic_table["Topic"].apply(
    lambda x: get_representative_titles(x, n=3)
)

final_topic_table["Proportion_all_documents"] = (
    final_topic_table["Count"] / n_documents * 100
)

final_topic_table["Proportion_valid_documents"] = (
    final_topic_table["Count"] / n_valid_documents * 100
)

final_topic_table = final_topic_table[
    [
        "Topic",
        "Count",
        "Proportion_all_documents",
        "Proportion_valid_documents",
        "Topic_label_raw",
        "Top_words",
        "Representative_titles"
    ]
].copy()

final_topic_table = final_topic_table.sort_values(
    "Count",
    ascending=False
).reset_index(drop=True)

final_topic_table.insert(
    0,
    "Rank",
    np.arange(1, len(final_topic_table) + 1)
)

final_topic_table["Proportion_all_documents"] = final_topic_table["Proportion_all_documents"].round(2)
final_topic_table["Proportion_valid_documents"] = final_topic_table["Proportion_valid_documents"].round(2)

display(final_topic_table.head(50))

# ------------------------------------------------------------
# Save final topic information
# ------------------------------------------------------------
final_topic_info.to_csv(
    FINAL_TOPIC_DIR / "final_topic_info_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

final_topic_table.to_csv(
    FINAL_TOPIC_DIR / "final_topic_table_for_manual_labeling.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Save final model
# ------------------------------------------------------------
final_model_save_path = FINAL_TOPIC_DIR / "bertopic_final_M3_more_conservative"

final_topic_model.save(
    final_model_save_path,
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=False
)

print()
print("Saved:")
print(FINAL_TOPIC_DIR / "final_document_topic_assignments.csv")
print(FINAL_TOPIC_DIR / "final_topic_info_raw.csv")
print(FINAL_TOPIC_DIR / "final_topic_table_for_manual_labeling.csv")
print(final_model_save_path)

In [ ]:
# ============================================================
# CELL 34 — Manual topic labeling and final Table 4 preparation
# Final BERTopic model: M3_more_conservative
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Manual topic labels
# ------------------------------------------------------------
manual_topic_labels = {
    0: "Porcine viral infections and viral replication",
    1: "Anthelmintic resistance and gastrointestinal nematodes",
    2: "Salmonella, Escherichia coli, and antimicrobial resistance",
    3: "Transboundary viral diseases and biosecurity",
    4: "Veterinary anaesthesia, analgesia, and sedation",
    5: "Neurology and diagnostic imaging",
    6: "Veterinary oncology and lymphoma",
    7: "Tick-borne and vector-borne diseases",
    8: "Rabies, dog population health, and veterinary education",
    9: "Dairy cattle lameness and calf health",
    10: "Bovine viral diseases",
    11: "Poultry production and health",
    12: "Fish health and aquaculture",
    13: "Equine endocrine and metabolic disorders",
    14: "Bovine tuberculosis and mycobacterial diseases",
    15: "Equine lameness, tendon injury, and racing medicine",
    16: "Toxoplasma gondii, Neospora caninum, and protozoal infections",
    17: "Dental and musculoskeletal disorders",
    18: "Renal and urinary disorders",
    19: "Cardiology and echocardiographic assessment",
    20: "Avian influenza and influenza viruses",
    21: "Bovine mastitis and udder health",
    22: "Rumen fermentation and cattle nutrition",
    23: "Companion-animal viral infections",
    24: "Porcine bacterial respiratory diseases",
    25: "Obesity, diabetes, and insulin-related metabolic disorders",
    26: "Piglet gut microbiota and weaning-related health",
    27: "Mycoplasma infections in animals",
    28: "Tick control and acaricidal activity",
    29: "Respiratory ventilation and airway management",
    30: "Ovarian function, oocytes, and embryo technologies",
    31: "Equine herpesvirus and West Nile virus",
    32: "Antimicrobial use and stewardship",
    33: "Pancreatitis and inflammatory bowel disease",
    34: "Hematology, transfusion, and sepsis",
    35: "Mesenchymal stem cells and regenerative medicine",
    36: "Brucellosis",
    37: "Semen quality and reproductive physiology",
    38: "Leishmaniasis",
    39: "Echinococcosis and cestode infections",
    40: "Cryptosporidium and Giardia infections",
    41: "Heartworm and Dirofilaria infections",
    42: "Methicillin-resistant Staphylococcus and antimicrobial resistance",
    43: "Equine airway disease and asthma",
    44: "Parasitic infections and zoonotic parasites",
    45: "Genetic traits and muscle development"
}

# ------------------------------------------------------------
# Broad thematic categories
# ------------------------------------------------------------
broad_topic_themes = {
    0: "Infectious disease and virology",
    1: "Parasitology",
    2: "Bacterial infection and antimicrobial resistance",
    3: "Infectious disease and biosecurity",
    4: "Clinical medicine and anaesthesia",
    5: "Clinical medicine and diagnostics",
    6: "Clinical medicine and oncology",
    7: "Parasitology and vector-borne disease",
    8: "Public health and education",
    9: "Production animal health",
    10: "Infectious disease and virology",
    11: "Poultry health and production",
    12: "Aquatic animal health",
    13: "Equine medicine",
    14: "Mycobacterial disease",
    15: "Equine medicine",
    16: "Parasitology",
    17: "Clinical medicine and pathology",
    18: "Clinical medicine",
    19: "Clinical medicine and cardiology",
    20: "Infectious disease and virology",
    21: "Production animal health",
    22: "Nutrition and production physiology",
    23: "Companion animal infectious disease",
    24: "Bacterial infection",
    25: "Metabolic disease",
    26: "Gut health and microbiota",
    27: "Bacterial infection",
    28: "Parasitology and vector control",
    29: "Clinical medicine and anaesthesia",
    30: "Reproduction and biotechnology",
    31: "Equine infectious disease",
    32: "Antimicrobial use and stewardship",
    33: "Clinical medicine and gastroenterology",
    34: "Clinical medicine and critical care",
    35: "Regenerative medicine",
    36: "Zoonotic bacterial disease",
    37: "Reproduction and physiology",
    38: "Parasitology and zoonosis",
    39: "Parasitology and zoonosis",
    40: "Parasitology",
    41: "Parasitology",
    42: "Antimicrobial resistance",
    43: "Equine medicine",
    44: "Parasitology and zoonosis",
    45: "Genetics and production traits"
}

# ------------------------------------------------------------
# Apply labels
# ------------------------------------------------------------
final_topic_table_labeled = final_topic_table.copy()

final_topic_table_labeled["Manual_topic_label"] = final_topic_table_labeled["Topic"].map(
    manual_topic_labels
)

final_topic_table_labeled["Broad_theme"] = final_topic_table_labeled["Topic"].map(
    broad_topic_themes
)

# ------------------------------------------------------------
# Check missing labels
# ------------------------------------------------------------
missing_label_topics = final_topic_table_labeled[
    final_topic_table_labeled["Manual_topic_label"].isna()
]["Topic"].tolist()

missing_theme_topics = final_topic_table_labeled[
    final_topic_table_labeled["Broad_theme"].isna()
]["Topic"].tolist()

print("Manual labeling check")
print("---------------------")
print("Missing manual labels:", missing_label_topics)
print("Missing broad themes: ", missing_theme_topics)

if len(missing_label_topics) > 0:
    raise ValueError(f"Missing manual labels for topics: {missing_label_topics}")

if len(missing_theme_topics) > 0:
    raise ValueError(f"Missing broad themes for topics: {missing_theme_topics}")

# ------------------------------------------------------------
# Main Table 4 for manuscript
# Use valid documents as denominator to avoid outlier-percentage confusion.
# ------------------------------------------------------------
table4_final_topics = final_topic_table_labeled[
    [
        "Rank",
        "Topic",
        "Manual_topic_label",
        "Broad_theme",
        "Count",
        "Proportion_valid_documents",
        "Top_words"
    ]
].copy()

table4_final_topics = table4_final_topics.rename(
    columns={
        "Topic": "Topic ID",
        "Manual_topic_label": "Topic label",
        "Broad_theme": "Broad theme",
        "Count": "Documents",
        "Proportion_valid_documents": "Proportion of valid documents (%)",
        "Top_words": "Representative keywords"
    }
)

table4_final_topics["Proportion of valid documents (%)"] = (
    table4_final_topics["Proportion of valid documents (%)"]
    .round(2)
)

# ------------------------------------------------------------
# Supplementary table with representative titles
# ------------------------------------------------------------
supplementary_final_topics = final_topic_table_labeled[
    [
        "Rank",
        "Topic",
        "Manual_topic_label",
        "Broad_theme",
        "Count",
        "Proportion_all_documents",
        "Proportion_valid_documents",
        "Top_words",
        "Representative_titles"
    ]
].copy()

supplementary_final_topics = supplementary_final_topics.rename(
    columns={
        "Topic": "Topic ID",
        "Manual_topic_label": "Topic label",
        "Broad_theme": "Broad theme",
        "Count": "Documents",
        "Proportion_all_documents": "Proportion of all documents (%)",
        "Proportion_valid_documents": "Proportion of valid documents (%)",
        "Top_words": "Representative keywords",
        "Representative_titles": "Representative titles"
    }
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------
display(table4_final_topics)
display(supplementary_final_topics.head(20))

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
table4_final_topics.to_csv(
    FINAL_TOPIC_DIR / "table4_final_bertopic_topics.csv",
    index=False,
    encoding="utf-8-sig"
)

supplementary_final_topics.to_csv(
    FINAL_TOPIC_DIR / "supplementary_final_bertopic_topics_with_representative_titles.csv",
    index=False,
    encoding="utf-8-sig"
)

# Also save labeled full table
final_topic_table_labeled.to_csv(
    FINAL_TOPIC_DIR / "final_topic_table_labeled_full.csv",
    index=False,
    encoding="utf-8-sig"
)

print()
print("Saved:")
print(FINAL_TOPIC_DIR / "table4_final_bertopic_topics.csv")
print(FINAL_TOPIC_DIR / "supplementary_final_bertopic_topics_with_representative_titles.csv")
print(FINAL_TOPIC_DIR / "final_topic_table_labeled_full.csv")

print()
print("Final topic table summary")
print("-------------------------")
print(f"Number of valid topics: {table4_final_topics['Topic ID'].nunique():,}")
print(f"Total valid documents:  {table4_final_topics['Documents'].sum():,}")
print(f"Percentage sum:         {table4_final_topics['Proportion of valid documents (%)'].sum():.2f}%")

In [ ]:
# ============================================================
# CELL 35 — Create Supplementary Table S1
# BERTopic model selection table
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# Load BERTopic parameter comparison table if not already loaded
# ------------------------------------------------------------
if "bertopic_parameter_comparison" not in globals():
    comparison_path = BERTOPIC_DIR / "bertopic_parameter_comparison.csv"
    
    if comparison_path.exists():
        bertopic_parameter_comparison = pd.read_csv(
            comparison_path,
            encoding="utf-8-sig"
        )
        print("Loaded:")
        print(comparison_path)
    else:
        raise FileNotFoundError(
            "bertopic_parameter_comparison was not found in memory "
            "and bertopic_parameter_comparison.csv does not exist."
        )

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
SUPP_TABLE_DIR = RESULTS_DIR / "supplementary_tables"
SUPP_TABLE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Selected final model
# ------------------------------------------------------------
selected_model = "M3_more_conservative"

# ------------------------------------------------------------
# Create clean Supplementary Table S1
# ------------------------------------------------------------
supp_table_s1 = bertopic_parameter_comparison.copy()

# Extract model order
supp_table_s1["Model_order"] = (
    supp_table_s1["Model"]
    .astype(str)
    .str.extract(r"M(\d+)")
    .astype(float)
)

# Add fixed information
supp_table_s1["Embedding model"] = "sentence-transformers/all-MiniLM-L6-v2"
supp_table_s1["Text input"] = "Title + abstract"

# Add selection decision
supp_table_s1["Selection"] = np.where(
    supp_table_s1["Model"] == selected_model,
    "Selected",
    "Not selected"
)

# Add short rationale
selection_rationale = {
    "M1_baseline": (
        "Not selected because it produced too many valid topics "
        "and showed a less stable topic-size distribution."
    ),
    "M2_larger_clusters": (
        "Not selected because it produced more valid topics than the selected model, "
        "despite a relatively low outlier ratio."
    ),
    "M3_more_conservative": (
        "Selected because it showed the lowest outlier ratio, "
        "a manageable number of valid topics, high topic diversity, "
        "and a stable topic-size distribution."
    ),
    "M4_lower_outlier": (
        "Not selected because it produced more valid topics and a higher outlier ratio "
        "than the selected model."
    ),
    "M5_stable_broader": (
        "Not selected because it produced a similar number of valid topics "
        "but had a higher outlier ratio than the selected model."
    ),
    "M6_high_min_df": (
        "Not selected because it produced a similar number of valid topics "
        "but had lower topic diversity and a higher outlier ratio than the selected model."
    )
}

supp_table_s1["Decision rationale"] = supp_table_s1["Model"].map(selection_rationale)

# ------------------------------------------------------------
# Select and rename columns for manuscript-ready table
# ------------------------------------------------------------
supp_table_s1 = supp_table_s1[
    [
        "Model_order",
        "Model",
        "Embedding model",
        "Text input",
        "Documents",
        "Valid_documents",
        "Outliers",
        "Outlier_ratio",
        "Raw_topics",
        "Topic_diversity",
        "Min_topic_size",
        "Median_topic_size",
        "Max_topic_size",
        "Topic_size_CV",
        "UMAP_n_neighbors",
        "UMAP_n_components",
        "UMAP_min_dist",
        "HDBSCAN_min_cluster_size",
        "HDBSCAN_min_samples",
        "Vectorizer_min_df",
        "Vectorizer_max_df",
        "Selection",
        "Decision rationale"
    ]
].copy()

supp_table_s1 = supp_table_s1.rename(
    columns={
        "Model_order": "Model order",
        "Model": "Model",
        "Documents": "Documents",
        "Valid_documents": "Valid documents",
        "Outliers": "Outliers",
        "Outlier_ratio": "Outlier ratio",
        "Raw_topics": "Valid topics",
        "Topic_diversity": "Topic diversity",
        "Min_topic_size": "Minimum topic size",
        "Median_topic_size": "Median topic size",
        "Max_topic_size": "Maximum topic size",
        "Topic_size_CV": "Topic-size CV",
        "UMAP_n_neighbors": "UMAP n_neighbors",
        "UMAP_n_components": "UMAP n_components",
        "UMAP_min_dist": "UMAP min_dist",
        "HDBSCAN_min_cluster_size": "HDBSCAN min_cluster_size",
        "HDBSCAN_min_samples": "HDBSCAN min_samples",
        "Vectorizer_min_df": "Vectorizer min_df",
        "Vectorizer_max_df": "Vectorizer max_df"
    }
)

# ------------------------------------------------------------
# Sort by model order
# ------------------------------------------------------------
supp_table_s1 = (
    supp_table_s1
    .sort_values("Model order")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Round numeric columns
# ------------------------------------------------------------
round_cols_4 = [
    "Outlier ratio",
    "Topic diversity",
    "Topic-size CV",
    "Vectorizer max_df"
]

for col in round_cols_4:
    if col in supp_table_s1.columns:
        supp_table_s1[col] = supp_table_s1[col].round(4)

round_cols_1 = [
    "Median topic size"
]

for col in round_cols_1:
    if col in supp_table_s1.columns:
        supp_table_s1[col] = supp_table_s1[col].round(1)

# Model order as integer
supp_table_s1["Model order"] = supp_table_s1["Model order"].astype(int)

# ------------------------------------------------------------
# Display final Supplementary Table S1
# ------------------------------------------------------------
display(supp_table_s1)

# ------------------------------------------------------------
# Save as CSV and Excel
# ------------------------------------------------------------
csv_path = SUPP_TABLE_DIR / "Supplementary_Table_S1_BERTopic_model_selection.csv"
xlsx_path = SUPP_TABLE_DIR / "Supplementary_Table_S1_BERTopic_model_selection.xlsx"
txt_path = SUPP_TABLE_DIR / "Supplementary_Table_S1_BERTopic_model_selection_caption_and_note.txt"

supp_table_s1.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

supp_table_s1.to_excel(
    xlsx_path,
    index=False
)

# ------------------------------------------------------------
# Caption and table note
# ------------------------------------------------------------
supp_table_s1_caption = (
    "Supplementary Table S1. Comparison of BERTopic parameter settings used for final model selection."
)

supp_table_s1_note = (
    "Note. Candidate BERTopic models were compared using the number of valid topics, "
    "outlier ratio, topic diversity, topic-size distribution, and semantic interpretability. "
    "Documents assigned to Topic -1 were treated as outliers. Topic-size CV indicates the "
    "coefficient of variation of topic sizes among valid topics. The M3_more_conservative "
    "model was selected as the final model because it showed the lowest outlier ratio, "
    "a manageable number of valid topics, high topic diversity, and a stable topic-size distribution."
)

with open(txt_path, "w", encoding="utf-8-sig") as f:
    f.write(supp_table_s1_caption)
    f.write("\n\n")
    f.write(supp_table_s1_note)

print()
print("Saved:")
print(csv_path)
print(xlsx_path)
print(txt_path)

print()
print("Caption:")
print(supp_table_s1_caption)

print()
print("Note:")
print(supp_table_s1_note)

In [ ]:
# ============================================================
# CELL 34B — Revise selected manual topic labels for Table 4
# No model refitting is required
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# Load labeled topic tables if not already available in memory
# ------------------------------------------------------------
if "FINAL_TOPIC_DIR" not in globals():
    FINAL_TOPIC_DIR = BERTOPIC_DIR / "final_model"

table4_path = FINAL_TOPIC_DIR / "table4_final_bertopic_topics.csv"
supp_path = FINAL_TOPIC_DIR / "supplementary_final_bertopic_topics_with_representative_titles.csv"
full_path = FINAL_TOPIC_DIR / "final_topic_table_labeled_full.csv"

if "table4_final_topics" not in globals():
    table4_final_topics = pd.read_csv(
        table4_path,
        encoding="utf-8-sig"
    )

if "supplementary_final_topics" not in globals():
    supplementary_final_topics = pd.read_csv(
        supp_path,
        encoding="utf-8-sig"
    )

if "final_topic_table_labeled" not in globals():
    final_topic_table_labeled = pd.read_csv(
        full_path,
        encoding="utf-8-sig"
    )

# ------------------------------------------------------------
# Revised topic labels
# ------------------------------------------------------------
revised_topic_labels = {
    5: "Neurology, spinal disorders, and diagnostic imaging",
    8: "Rabies, animal-bite prevention, and veterinary education",
    17: "Dental, oral, and orthopedic disorders",
    28: "Ectoparasite control and acaricidal activity",
    44: "Zoonotic and companion-animal parasitic infections",
    45: "Genomic traits, growth, and production performance"
}

# ------------------------------------------------------------
# Revised broad themes
# ------------------------------------------------------------
revised_broad_themes = {
    5: "Clinical medicine and diagnostics",
    8: "Veterinary public health and education",
    17: "Clinical medicine and orthopedics",
    28: "Parasitology and vector control",
    44: "Parasitology and zoonosis",
    45: "Genetics and production traits"
}

# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------
def apply_revised_labels(df, topic_col, label_col, theme_col):
    df = df.copy()

    for topic_id, new_label in revised_topic_labels.items():
        df.loc[df[topic_col] == topic_id, label_col] = new_label

    for topic_id, new_theme in revised_broad_themes.items():
        df.loc[df[topic_col] == topic_id, theme_col] = new_theme

    return df

# ------------------------------------------------------------
# Apply revisions to each table
# ------------------------------------------------------------
table4_final_topics_revised = apply_revised_labels(
    table4_final_topics,
    topic_col="Topic ID",
    label_col="Topic label",
    theme_col="Broad theme"
)

supplementary_final_topics_revised = apply_revised_labels(
    supplementary_final_topics,
    topic_col="Topic ID",
    label_col="Topic label",
    theme_col="Broad theme"
)

final_topic_table_labeled_revised = apply_revised_labels(
    final_topic_table_labeled,
    topic_col="Topic",
    label_col="Manual_topic_label",
    theme_col="Broad_theme"
)

# ------------------------------------------------------------
# Display revised rows only
# ------------------------------------------------------------
print("Revised Table 4 labels")
print("----------------------")

display(
    table4_final_topics_revised[
        table4_final_topics_revised["Topic ID"].isin(revised_topic_labels.keys())
    ][
        [
            "Rank",
            "Topic ID",
            "Topic label",
            "Broad theme",
            "Documents",
            "Proportion of valid documents (%)",
            "Representative keywords"
        ]
    ]
)

# ------------------------------------------------------------
# Save revised versions
# ------------------------------------------------------------
table4_revised_path = FINAL_TOPIC_DIR / "table4_final_bertopic_topics_revised.csv"
supp_revised_path = FINAL_TOPIC_DIR / "supplementary_final_bertopic_topics_with_representative_titles_revised.csv"
full_revised_path = FINAL_TOPIC_DIR / "final_topic_table_labeled_full_revised.csv"

table4_final_topics_revised.to_csv(
    table4_revised_path,
    index=False,
    encoding="utf-8-sig"
)

supplementary_final_topics_revised.to_csv(
    supp_revised_path,
    index=False,
    encoding="utf-8-sig"
)

final_topic_table_labeled_revised.to_csv(
    full_revised_path,
    index=False,
    encoding="utf-8-sig"
)

print()
print("Saved:")
print(table4_revised_path)
print(supp_revised_path)
print(full_revised_path)

In [ ]:
# ============================================================
# CELL 36 — Figure 5A–C
# Distribution of final BERTopic-derived topics
#
# Figure 5A: Broad thematic distribution by document proportion
# Figure 5B: Number of topics within each broad theme
# Figure 5C: Top 15 individual BERTopic topics
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import font_manager
from pathlib import Path
import textwrap
import math

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
if "FINAL_TOPIC_DIR" not in globals():
    FINAL_TOPIC_DIR = BERTOPIC_DIR / "final_model"

FIGURE_DIR = RESULTS_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TABLE_DIR = RESULTS_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

table4_path = FINAL_TOPIC_DIR / "table4_final_bertopic_topics_revised.csv"

if not table4_path.exists():
    table4_path = FINAL_TOPIC_DIR / "table4_final_bertopic_topics.csv"

if not table4_path.exists():
    raise FileNotFoundError(
        "Table 4 topic file was not found. "
        "Please check FINAL_TOPIC_DIR and the Table 4 CSV file."
    )

# ------------------------------------------------------------
# Load Table 4
# ------------------------------------------------------------
table4 = pd.read_csv(table4_path, encoding="utf-8-sig")

print("Loaded:")
print(table4_path)
print()
print("Columns:")
print(table4.columns.tolist())

# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------
required_columns = [
    "Rank",
    "Topic ID",
    "Topic label",
    "Broad theme",
    "Documents",
    "Proportion of valid documents (%)",
    "Representative keywords"
]

missing_columns = [
    col for col in required_columns
    if col not in table4.columns
]

if len(missing_columns) > 0:
    raise ValueError(f"Missing columns in Table 4 file: {missing_columns}")

# ------------------------------------------------------------
# Numeric formatting
# ------------------------------------------------------------
table4["Topic ID"] = table4["Topic ID"].astype(int)
table4["Documents"] = pd.to_numeric(table4["Documents"], errors="coerce")
table4["Proportion of valid documents (%)"] = pd.to_numeric(
    table4["Proportion of valid documents (%)"],
    errors="coerce"
)

valid_document_total = table4["Documents"].sum()

print()
print("Topic table summary")
print("-------------------")
print(f"Number of valid topics: {table4['Topic ID'].nunique():,}")
print(f"Total valid documents:  {valid_document_total:,.0f}")
print(f"Percentage sum:         {table4['Proportion of valid documents (%)'].sum():.2f}%")

# ------------------------------------------------------------
# Figure style
# ------------------------------------------------------------
plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.unicode_minus"] = False

actual_font = font_manager.findfont("Arial", fallback_to_default=True)
print()
print("Actual font used:")
print(actual_font)

def wrap_label(text, width=34):
    return "\n".join(textwrap.wrap(str(text), width=width))

def get_nice_xmax(max_value, tick_interval):
    return math.ceil(max_value / tick_interval) * tick_interval

def style_axis(ax, grid_axis="x"):
    ax.tick_params(
        axis="both",
        direction="out",
        length=4,
        width=0.9,
        labelsize=10
    )
    
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.9)
    ax.spines["bottom"].set_linewidth(0.9)
    
    ax.grid(
        axis=grid_axis,
        linestyle="-",
        linewidth=0.4,
        alpha=0.25
    )
    
    ax.set_axisbelow(True)

# ------------------------------------------------------------
# Higher-level theme groups
# ------------------------------------------------------------
theme_group_mapping = {
    0: "Infectious disease,\nvirology, and AMR",
    1: "Parasitology and\nvector-borne disease",
    2: "Infectious disease,\nvirology, and AMR",
    3: "Infectious disease,\nvirology, and AMR",
    4: "Clinical medicine,\ndiagnostics, and therapeutics",
    5: "Clinical medicine,\ndiagnostics, and therapeutics",
    6: "Clinical medicine,\ndiagnostics, and therapeutics",
    7: "Parasitology and\nvector-borne disease",
    8: "Veterinary public health\nand education",
    9: "Production animal health,\nnutrition, and physiology",
    10: "Infectious disease,\nvirology, and AMR",
    11: "Production animal health,\nnutrition, and physiology",
    12: "Aquatic animal health",
    13: "Clinical medicine,\ndiagnostics, and therapeutics",
    14: "Infectious disease,\nvirology, and AMR",
    15: "Clinical medicine,\ndiagnostics, and therapeutics",
    16: "Parasitology and\nvector-borne disease",
    17: "Clinical medicine,\ndiagnostics, and therapeutics",
    18: "Clinical medicine,\ndiagnostics, and therapeutics",
    19: "Clinical medicine,\ndiagnostics, and therapeutics",
    20: "Infectious disease,\nvirology, and AMR",
    21: "Production animal health,\nnutrition, and physiology",
    22: "Production animal health,\nnutrition, and physiology",
    23: "Infectious disease,\nvirology, and AMR",
    24: "Infectious disease,\nvirology, and AMR",
    25: "Clinical medicine,\ndiagnostics, and therapeutics",
    26: "Production animal health,\nnutrition, and physiology",
    27: "Infectious disease,\nvirology, and AMR",
    28: "Parasitology and\nvector-borne disease",
    29: "Clinical medicine,\ndiagnostics, and therapeutics",
    30: "Reproduction and\nbiotechnology",
    31: "Infectious disease,\nvirology, and AMR",
    32: "Infectious disease,\nvirology, and AMR",
    33: "Clinical medicine,\ndiagnostics, and therapeutics",
    34: "Clinical medicine,\ndiagnostics, and therapeutics",
    35: "Clinical medicine,\ndiagnostics, and therapeutics",
    36: "Infectious disease,\nvirology, and AMR",
    37: "Reproduction and\nbiotechnology",
    38: "Parasitology and\nvector-borne disease",
    39: "Parasitology and\nvector-borne disease",
    40: "Parasitology and\nvector-borne disease",
    41: "Parasitology and\nvector-borne disease",
    42: "Infectious disease,\nvirology, and AMR",
    43: "Clinical medicine,\ndiagnostics, and therapeutics",
    44: "Parasitology and\nvector-borne disease",
    45: "Production animal health,\nnutrition, and physiology"
}

fig5_df = table4.copy()
fig5_df["Theme group"] = fig5_df["Topic ID"].map(theme_group_mapping)

missing_theme_group = fig5_df[fig5_df["Theme group"].isna()]["Topic ID"].tolist()

if len(missing_theme_group) > 0:
    raise ValueError(f"Missing theme group mapping for Topic IDs: {missing_theme_group}")

theme_summary = (
    fig5_df
    .groupby("Theme group", as_index=False)
    .agg(
        Documents=("Documents", "sum"),
        Number_of_topics=("Topic ID", "nunique")
    )
)

theme_summary["Proportion of valid documents (%)"] = (
    theme_summary["Documents"] / valid_document_total * 100
)

theme_summary = (
    theme_summary
    .sort_values("Documents", ascending=False)
    .reset_index(drop=True)
)

theme_summary["Rank"] = np.arange(1, len(theme_summary) + 1)

theme_summary_display = theme_summary[
    [
        "Rank",
        "Theme group",
        "Number_of_topics",
        "Documents",
        "Proportion of valid documents (%)"
    ]
].copy()

theme_summary_display["Proportion of valid documents (%)"] = (
    theme_summary_display["Proportion of valid documents (%)"].round(2)
)

display(theme_summary_display)

theme_summary_csv = TABLE_DIR / "figure_5_broad_theme_summary.csv"

theme_summary_display.to_csv(
    theme_summary_csv,
    index=False,
    encoding="utf-8-sig"
)

print()
print("Saved theme summary:")
print(theme_summary_csv)

# ============================================================
# Figure 5A — Broad thematic distribution by document proportion
# ============================================================

fig5a_df = theme_summary_display.sort_values(
    "Proportion of valid documents (%)",
    ascending=True
).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(5.8, 3.7))

y_pos = np.arange(len(fig5a_df))
bar_color = "#E6B8CF"

ax.barh(
    y_pos,
    fig5a_df["Proportion of valid documents (%)"],
    height=0.68,
    color=bar_color,
    edgecolor="black",
    linewidth=0.5,
    zorder=3
)

ax.set_yticks(y_pos)
ax.set_yticklabels(fig5a_df["Theme group"], fontsize=9.5)

x_tick_interval = 10
xmax = get_nice_xmax(fig5a_df["Proportion of valid documents (%)"].max(), x_tick_interval)

ax.set_xlim(0, xmax * 1.20)
ax.xaxis.set_major_locator(ticker.MultipleLocator(x_tick_interval))
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"{x:.0f}"))

ax.set_xlabel(
    "Proportion of valid documents (%)",
    fontsize=13,
    fontweight="bold"
)

style_axis(ax)

for i, row in fig5a_df.iterrows():
    ax.text(
        row["Proportion of valid documents (%)"] + xmax * 0.015,
        i,
        f"{row['Proportion of valid documents (%)']:.2f}%",
        va="center",
        ha="left",
        fontsize=8.8
    )

plt.tight_layout()

fig5a_png = FIGURE_DIR / "figure_5a_broad_theme_proportion.png"
fig5a_svg = FIGURE_DIR / "figure_5a_broad_theme_proportion.svg"

plt.savefig(fig5a_png, dpi=600, bbox_inches="tight")
plt.savefig(fig5a_svg, bbox_inches="tight")

plt.show()

print()
print("Saved Figure 5A:")
print(fig5a_png)
print(fig5a_svg)

# ============================================================
# Figure 5B — Number of topics within each broad theme
# ============================================================

fig5b_df = theme_summary_display.sort_values(
    "Number_of_topics",
    ascending=True
).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(5.8, 3.7))

y_pos = np.arange(len(fig5b_df))
bar_color = "#D9D9D9"

ax.barh(
    y_pos,
    fig5b_df["Number_of_topics"],
    height=0.68,
    color=bar_color,
    edgecolor="black",
    linewidth=0.5,
    zorder=3
)

ax.set_yticks(y_pos)
ax.set_yticklabels(fig5b_df["Theme group"], fontsize=9.5)

x_tick_interval = 2
xmax = get_nice_xmax(fig5b_df["Number_of_topics"].max(), x_tick_interval)

ax.set_xlim(0, xmax * 1.25)
ax.xaxis.set_major_locator(ticker.MultipleLocator(x_tick_interval))
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"{int(x)}"))

ax.set_xlabel(
    "Number of topics",
    fontsize=13,
    fontweight="bold"
)

style_axis(ax)

for i, row in fig5b_df.iterrows():
    ax.text(
        row["Number_of_topics"] + xmax * 0.02,
        i,
        f"{int(row['Number_of_topics'])}",
        va="center",
        ha="left",
        fontsize=8.8
    )

plt.tight_layout()

fig5b_png = FIGURE_DIR / "figure_5b_number_of_topics_by_theme.png"
fig5b_svg = FIGURE_DIR / "figure_5b_number_of_topics_by_theme.svg"

plt.savefig(fig5b_png, dpi=600, bbox_inches="tight")
plt.savefig(fig5b_svg, bbox_inches="tight")

plt.show()

print()
print("Saved Figure 5B:")
print(fig5b_png)
print(fig5b_svg)

# ============================================================
# Figure 5C — Top 15 individual BERTopic topics
# Dot plot instead of large bar plot
# ============================================================

top_n_topics = 15

fig5c_df = (
    table4
    .sort_values("Documents", ascending=False)
    .head(top_n_topics)
    .copy()
)

fig5c_df["Topic display label"] = fig5c_df.apply(
    lambda row: f"T{int(row['Topic ID'])}: {row['Topic label']}",
    axis=1
)

fig5c_df["Topic display label"] = fig5c_df["Topic display label"].apply(
    lambda x: wrap_label(x, width=36)
)

fig5c_df = fig5c_df.sort_values("Documents", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(5.8, 5.2))

y_pos = np.arange(len(fig5c_df))
line_color = "#BDBDBD"
point_color = "#E6B8CF"

for i, row in fig5c_df.iterrows():
    ax.hlines(
        y=i,
        xmin=0,
        xmax=row["Documents"],
        color=line_color,
        linewidth=1.0,
        zorder=2
    )

ax.scatter(
    fig5c_df["Documents"],
    y_pos,
    s=55,
    color=point_color,
    edgecolor="black",
    linewidth=0.6,
    zorder=3
)

ax.set_yticks(y_pos)
ax.set_yticklabels(fig5c_df["Topic display label"], fontsize=8.8)

x_tick_interval = 500
xmax = get_nice_xmax(fig5c_df["Documents"].max(), x_tick_interval)

ax.set_xlim(0, xmax * 1.22)
ax.xaxis.set_major_locator(ticker.MultipleLocator(x_tick_interval))
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"{int(x):,}"))

ax.set_xlabel(
    "Number of documents",
    fontsize=13,
    fontweight="bold"
)

style_axis(ax)

for i, row in fig5c_df.iterrows():
    ax.text(
        row["Documents"] + xmax * 0.015,
        i,
        f"{int(row['Documents']):,} ({row['Proportion of valid documents (%)']:.2f}%)",
        va="center",
        ha="left",
        fontsize=8.3
    )

plt.tight_layout()

fig5c_png = FIGURE_DIR / "figure_5c_top15_individual_topics_dotplot.png"
fig5c_svg = FIGURE_DIR / "figure_5c_top15_individual_topics_dotplot.svg"

plt.savefig(fig5c_png, dpi=600, bbox_inches="tight")
plt.savefig(fig5c_svg, bbox_inches="tight")

plt.show()

print()
print("Saved Figure 5C:")
print(fig5c_png)
print(fig5c_svg)

# ============================================================
# Figure 5 combined — A/B/C in one file
# ============================================================

fig = plt.figure(figsize=(7.0, 9.6))

gs = fig.add_gridspec(
    nrows=3,
    ncols=1,
    height_ratios=[1.0, 1.0, 1.45],
    hspace=0.55
)

# ------------------------------------------------------------
# Panel A
# ------------------------------------------------------------
ax1 = fig.add_subplot(gs[0, 0])

fig5a_plot = fig5a_df.copy()
y_pos = np.arange(len(fig5a_plot))

ax1.barh(
    y_pos,
    fig5a_plot["Proportion of valid documents (%)"],
    height=0.68,
    color="#E6B8CF",
    edgecolor="black",
    linewidth=0.5,
    zorder=3
)

ax1.set_yticks(y_pos)
ax1.set_yticklabels(fig5a_plot["Theme group"], fontsize=8.5)

ax1.set_xlim(0, xmax= get_nice_xmax(fig5a_plot["Proportion of valid documents (%)"].max(), 10) * 1.20)
ax1.xaxis.set_major_locator(ticker.MultipleLocator(10))
ax1.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"{x:.0f}"))

ax1.set_xlabel(
    "Proportion of valid documents (%)",
    fontsize=11,
    fontweight="bold"
)

style_axis(ax1)

for i, row in fig5a_plot.iterrows():
    ax1.text(
        row["Proportion of valid documents (%)"] + get_nice_xmax(fig5a_plot["Proportion of valid documents (%)"].max(), 10) * 0.015,
        i,
        f"{row['Proportion of valid documents (%)']:.1f}%",
        va="center",
        ha="left",
        fontsize=7.8
    )

ax1.text(
    -0.18,
    1.06,
    "A",
    transform=ax1.transAxes,
    fontsize=13,
    fontweight="bold",
    va="bottom",
    ha="left"
)

# ------------------------------------------------------------
# Panel B
# ------------------------------------------------------------
ax2 = fig.add_subplot(gs[1, 0])

fig5b_plot = fig5b_df.copy()
y_pos = np.arange(len(fig5b_plot))

ax2.barh(
    y_pos,
    fig5b_plot["Number_of_topics"],
    height=0.68,
    color="#D9D9D9",
    edgecolor="black",
    linewidth=0.5,
    zorder=3
)

ax2.set_yticks(y_pos)
ax2.set_yticklabels(fig5b_plot["Theme group"], fontsize=8.5)

topic_xmax = get_nice_xmax(fig5b_plot["Number_of_topics"].max(), 2)

ax2.set_xlim(0, topic_xmax * 1.25)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(2))
ax2.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"{int(x)}"))

ax2.set_xlabel(
    "Number of topics",
    fontsize=11,
    fontweight="bold"
)

style_axis(ax2)

for i, row in fig5b_plot.iterrows():
    ax2.text(
        row["Number_of_topics"] + topic_xmax * 0.02,
        i,
        f"{int(row['Number_of_topics'])}",
        va="center",
        ha="left",
        fontsize=7.8
    )

ax2.text(
    -0.18,
    1.06,
    "B",
    transform=ax2.transAxes,
    fontsize=13,
    fontweight="bold",
    va="bottom",
    ha="left"
)

# ------------------------------------------------------------
# Panel C
# ------------------------------------------------------------
ax3 = fig.add_subplot(gs[2, 0])

fig5c_plot = fig5c_df.copy()
y_pos = np.arange(len(fig5c_plot))

for i, row in fig5c_plot.iterrows():
    ax3.hlines(
        y=i,
        xmin=0,
        xmax=row["Documents"],
        color="#BDBDBD",
        linewidth=1.0,
        zorder=2
    )

ax3.scatter(
    fig5c_plot["Documents"],
    y_pos,
    s=45,
    color="#E6B8CF",
    edgecolor="black",
    linewidth=0.6,
    zorder=3
)

ax3.set_yticks(y_pos)
ax3.set_yticklabels(fig5c_plot["Topic display label"], fontsize=7.6)

topic_doc_xmax = get_nice_xmax(fig5c_plot["Documents"].max(), 500)

ax3.set_xlim(0, topic_doc_xmax * 1.22)
ax3.xaxis.set_major_locator(ticker.MultipleLocator(500))
ax3.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"{int(x):,}"))

ax3.set_xlabel(
    "Number of documents",
    fontsize=11,
    fontweight="bold"
)

style_axis(ax3)

for i, row in fig5c_plot.iterrows():
    ax3.text(
        row["Documents"] + topic_doc_xmax * 0.015,
        i,
        f"{int(row['Documents']):,} ({row['Proportion of valid documents (%)']:.1f}%)",
        va="center",
        ha="left",
        fontsize=7.2
    )

ax3.text(
    -0.18,
    1.04,
    "C",
    transform=ax3.transAxes,
    fontsize=13,
    fontweight="bold",
    va="bottom",
    ha="left"
)

plt.tight_layout()

fig5_combined_png = FIGURE_DIR / "figure_5_combined_topic_distribution_ABC.png"
fig5_combined_svg = FIGURE_DIR / "figure_5_combined_topic_distribution_ABC.svg"

plt.savefig(fig5_combined_png, dpi=600, bbox_inches="tight")
plt.savefig(fig5_combined_svg, bbox_inches="tight")

plt.show()

print()
print("Saved Figure 5 combined:")
print(fig5_combined_png)
print(fig5_combined_svg)

In [ ]:
# ============================================================
# CELL 36B — Revise Figure 5C and combined Figure 5
# More vertical space for Panel C and labels moved further right
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import font_manager
from pathlib import Path
import textwrap
import math

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
if "FINAL_TOPIC_DIR" not in globals():
    FINAL_TOPIC_DIR = BERTOPIC_DIR / "final_model"

FIGURE_DIR = RESULTS_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TABLE_DIR = RESULTS_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

table4_path = FINAL_TOPIC_DIR / "table4_final_bertopic_topics_revised.csv"

if not table4_path.exists():
    table4_path = FINAL_TOPIC_DIR / "table4_final_bertopic_topics.csv"

if not table4_path.exists():
    raise FileNotFoundError("Table 4 topic file was not found.")

# ------------------------------------------------------------
# Load Table 4
# ------------------------------------------------------------
table4 = pd.read_csv(table4_path, encoding="utf-8-sig")

table4["Topic ID"] = table4["Topic ID"].astype(int)
table4["Documents"] = pd.to_numeric(table4["Documents"], errors="coerce")
table4["Proportion of valid documents (%)"] = pd.to_numeric(
    table4["Proportion of valid documents (%)"],
    errors="coerce"
)

valid_document_total = table4["Documents"].sum()

# ------------------------------------------------------------
# Figure style
# ------------------------------------------------------------
plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.unicode_minus"] = False

actual_font = font_manager.findfont("Arial", fallback_to_default=True)
print("Actual font used:")
print(actual_font)

def wrap_label(text, width=34):
    return "\n".join(textwrap.wrap(str(text), width=width))

def get_nice_xmax(max_value, tick_interval):
    return math.ceil(max_value / tick_interval) * tick_interval

def style_axis(ax, grid_axis="x"):
    ax.tick_params(
        axis="both",
        direction="out",
        length=4,
        width=0.9,
        labelsize=10
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.9)
    ax.spines["bottom"].set_linewidth(0.9)

    ax.grid(
        axis=grid_axis,
        linestyle="-",
        linewidth=0.4,
        alpha=0.25
    )

    ax.set_axisbelow(True)

# ------------------------------------------------------------
# Higher-level theme groups
# ------------------------------------------------------------
theme_group_mapping = {
    0: "Infectious disease,\nvirology, and AMR",
    1: "Parasitology and\nvector-borne disease",
    2: "Infectious disease,\nvirology, and AMR",
    3: "Infectious disease,\nvirology, and AMR",
    4: "Clinical medicine,\ndiagnostics, and therapeutics",
    5: "Clinical medicine,\ndiagnostics, and therapeutics",
    6: "Clinical medicine,\ndiagnostics, and therapeutics",
    7: "Parasitology and\nvector-borne disease",
    8: "Veterinary public health\nand education",
    9: "Production animal health,\nnutrition, and physiology",
    10: "Infectious disease,\nvirology, and AMR",
    11: "Production animal health,\nnutrition, and physiology",
    12: "Aquatic animal health",
    13: "Clinical medicine,\ndiagnostics, and therapeutics",
    14: "Infectious disease,\nvirology, and AMR",
    15: "Clinical medicine,\ndiagnostics, and therapeutics",
    16: "Parasitology and\nvector-borne disease",
    17: "Clinical medicine,\ndiagnostics, and therapeutics",
    18: "Clinical medicine,\ndiagnostics, and therapeutics",
    19: "Clinical medicine,\ndiagnostics, and therapeutics",
    20: "Infectious disease,\nvirology, and AMR",
    21: "Production animal health,\nnutrition, and physiology",
    22: "Production animal health,\nnutrition, and physiology",
    23: "Infectious disease,\nvirology, and AMR",
    24: "Infectious disease,\nvirology, and AMR",
    25: "Clinical medicine,\ndiagnostics, and therapeutics",
    26: "Production animal health,\nnutrition, and physiology",
    27: "Infectious disease,\nvirology, and AMR",
    28: "Parasitology and\nvector-borne disease",
    29: "Clinical medicine,\ndiagnostics, and therapeutics",
    30: "Reproduction and\nbiotechnology",
    31: "Infectious disease,\nvirology, and AMR",
    32: "Infectious disease,\nvirology, and AMR",
    33: "Clinical medicine,\ndiagnostics, and therapeutics",
    34: "Clinical medicine,\ndiagnostics, and therapeutics",
    35: "Clinical medicine,\ndiagnostics, and therapeutics",
    36: "Infectious disease,\nvirology, and AMR",
    37: "Reproduction and\nbiotechnology",
    38: "Parasitology and\nvector-borne disease",
    39: "Parasitology and\nvector-borne disease",
    40: "Parasitology and\nvector-borne disease",
    41: "Parasitology and\nvector-borne disease",
    42: "Infectious disease,\nvirology, and AMR",
    43: "Clinical medicine,\ndiagnostics, and therapeutics",
    44: "Parasitology and\nvector-borne disease",
    45: "Production animal health,\nnutrition, and physiology"
}

fig5_df = table4.copy()
fig5_df["Theme group"] = fig5_df["Topic ID"].map(theme_group_mapping)

theme_summary = (
    fig5_df
    .groupby("Theme group", as_index=False)
    .agg(
        Documents=("Documents", "sum"),
        Number_of_topics=("Topic ID", "nunique")
    )
)

theme_summary["Proportion of valid documents (%)"] = (
    theme_summary["Documents"] / valid_document_total * 100
)

theme_summary = (
    theme_summary
    .sort_values("Documents", ascending=False)
    .reset_index(drop=True)
)

theme_summary["Rank"] = np.arange(1, len(theme_summary) + 1)

theme_summary_display = theme_summary[
    [
        "Rank",
        "Theme group",
        "Number_of_topics",
        "Documents",
        "Proportion of valid documents (%)"
    ]
].copy()

theme_summary_display["Proportion of valid documents (%)"] = (
    theme_summary_display["Proportion of valid documents (%)"].round(2)
)

# ------------------------------------------------------------
# Prepare A, B, C data
# ------------------------------------------------------------
fig5a_df = theme_summary_display.sort_values(
    "Proportion of valid documents (%)",
    ascending=True
).reset_index(drop=True)

fig5b_df = theme_summary_display.sort_values(
    "Number_of_topics",
    ascending=True
).reset_index(drop=True)

top_n_topics = 15

fig5c_df = (
    table4
    .sort_values("Documents", ascending=False)
    .head(top_n_topics)
    .copy()
)

fig5c_df["Topic display label"] = fig5c_df.apply(
    lambda row: f"T{int(row['Topic ID'])}: {row['Topic label']}",
    axis=1
)

fig5c_df["Topic display label"] = fig5c_df["Topic display label"].apply(
    lambda x: wrap_label(x, width=34)
)

fig5c_df = fig5c_df.sort_values("Documents", ascending=True).reset_index(drop=True)

# ============================================================
# Revised Figure 5C only
# ============================================================

fig, ax = plt.subplots(figsize=(6.2, 6.6))

y_pos = np.arange(len(fig5c_df))

line_color = "#BDBDBD"
point_color = "#E6B8CF"

for i, row in fig5c_df.iterrows():
    ax.hlines(
        y=i,
        xmin=0,
        xmax=row["Documents"],
        color=line_color,
        linewidth=1.0,
        zorder=2
    )

ax.scatter(
    fig5c_df["Documents"],
    y_pos,
    s=60,
    color=point_color,
    edgecolor="black",
    linewidth=0.6,
    zorder=3
)

ax.set_yticks(y_pos)
ax.set_yticklabels(fig5c_df["Topic display label"], fontsize=8.8)

topic_doc_xmax = get_nice_xmax(fig5c_df["Documents"].max(), 500)

# More right-side space for value labels
ax.set_xlim(0, topic_doc_xmax * 1.40)

ax.xaxis.set_major_locator(ticker.MultipleLocator(500))
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"{int(x):,}"))

ax.set_xlabel(
    "Number of documents",
    fontsize=13,
    fontweight="bold"
)

style_axis(ax)

# Move value labels further right
for i, row in fig5c_df.iterrows():
    ax.text(
        row["Documents"] + topic_doc_xmax * 0.045,
        i,
        f"{int(row['Documents']):,} ({row['Proportion of valid documents (%)']:.2f}%)",
        va="center",
        ha="left",
        fontsize=8.4
    )

plt.tight_layout()

fig5c_png = FIGURE_DIR / "figure_5c_top15_individual_topics_dotplot_revised.png"
fig5c_svg = FIGURE_DIR / "figure_5c_top15_individual_topics_dotplot_revised.svg"

plt.savefig(fig5c_png, dpi=600, bbox_inches="tight")
plt.savefig(fig5c_svg, bbox_inches="tight")

plt.show()

print()
print("Saved revised Figure 5C:")
print(fig5c_png)
print(fig5c_svg)

# ============================================================
# Revised combined Figure 5 — more vertical space for Panel C
# ============================================================

fig = plt.figure(figsize=(7.2, 11.2))

gs = fig.add_gridspec(
    nrows=3,
    ncols=1,
    height_ratios=[1.0, 1.0, 1.95],
    hspace=0.62
)

# ------------------------------------------------------------
# Panel A
# ------------------------------------------------------------
ax1 = fig.add_subplot(gs[0, 0])

fig5a_plot = fig5a_df.copy()
y_pos = np.arange(len(fig5a_plot))

theme_prop_xmax = get_nice_xmax(
    fig5a_plot["Proportion of valid documents (%)"].max(),
    10
)

ax1.barh(
    y_pos,
    fig5a_plot["Proportion of valid documents (%)"],
    height=0.68,
    color="#E6B8CF",
    edgecolor="black",
    linewidth=0.5,
    zorder=3
)

ax1.set_yticks(y_pos)
ax1.set_yticklabels(fig5a_plot["Theme group"], fontsize=8.5)

ax1.set_xlim(0, theme_prop_xmax * 1.20)
ax1.xaxis.set_major_locator(ticker.MultipleLocator(10))
ax1.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"{x:.0f}"))

ax1.set_xlabel(
    "Proportion of valid documents (%)",
    fontsize=11,
    fontweight="bold"
)

style_axis(ax1)

for i, row in fig5a_plot.iterrows():
    ax1.text(
        row["Proportion of valid documents (%)"] + theme_prop_xmax * 0.015,
        i,
        f"{row['Proportion of valid documents (%)']:.1f}%",
        va="center",
        ha="left",
        fontsize=7.8
    )

ax1.text(
    -0.18,
    1.06,
    "A",
    transform=ax1.transAxes,
    fontsize=13,
    fontweight="bold",
    va="bottom",
    ha="left"
)

# ------------------------------------------------------------
# Panel B
# ------------------------------------------------------------
ax2 = fig.add_subplot(gs[1, 0])

fig5b_plot = fig5b_df.copy()
y_pos = np.arange(len(fig5b_plot))

topic_xmax = get_nice_xmax(
    fig5b_plot["Number_of_topics"].max(),
    2
)

ax2.barh(
    y_pos,
    fig5b_plot["Number_of_topics"],
    height=0.68,
    color="#D9D9D9",
    edgecolor="black",
    linewidth=0.5,
    zorder=3
)

ax2.set_yticks(y_pos)
ax2.set_yticklabels(fig5b_plot["Theme group"], fontsize=8.5)

ax2.set_xlim(0, topic_xmax * 1.25)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(2))
ax2.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"{int(x)}"))

ax2.set_xlabel(
    "Number of topics",
    fontsize=11,
    fontweight="bold"
)

style_axis(ax2)

for i, row in fig5b_plot.iterrows():
    ax2.text(
        row["Number_of_topics"] + topic_xmax * 0.02,
        i,
        f"{int(row['Number_of_topics'])}",
        va="center",
        ha="left",
        fontsize=7.8
    )

ax2.text(
    -0.18,
    1.06,
    "B",
    transform=ax2.transAxes,
    fontsize=13,
    fontweight="bold",
    va="bottom",
    ha="left"
)

# ------------------------------------------------------------
# Panel C
# ------------------------------------------------------------
ax3 = fig.add_subplot(gs[2, 0])

fig5c_plot = fig5c_df.copy()
y_pos = np.arange(len(fig5c_plot))

for i, row in fig5c_plot.iterrows():
    ax3.hlines(
        y=i,
        xmin=0,
        xmax=row["Documents"],
        color="#BDBDBD",
        linewidth=1.0,
        zorder=2
    )

ax3.scatter(
    fig5c_plot["Documents"],
    y_pos,
    s=55,
    color="#E6B8CF",
    edgecolor="black",
    linewidth=0.6,
    zorder=3
)

ax3.set_yticks(y_pos)
ax3.set_yticklabels(fig5c_plot["Topic display label"], fontsize=7.8)

topic_doc_xmax = get_nice_xmax(
    fig5c_plot["Documents"].max(),
    500
)

# More right-side space for Panel C
ax3.set_xlim(0, topic_doc_xmax * 1.42)

ax3.xaxis.set_major_locator(ticker.MultipleLocator(500))
ax3.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"{int(x):,}"))

ax3.set_xlabel(
    "Number of documents",
    fontsize=11,
    fontweight="bold"
)

style_axis(ax3)

# Move labels further right
for i, row in fig5c_plot.iterrows():
    ax3.text(
        row["Documents"] + topic_doc_xmax * 0.050,
        i,
        f"{int(row['Documents']):,} ({row['Proportion of valid documents (%)']:.1f}%)",
        va="center",
        ha="left",
        fontsize=7.4
    )

ax3.text(
    -0.18,
    1.04,
    "C",
    transform=ax3.transAxes,
    fontsize=13,
    fontweight="bold",
    va="bottom",
    ha="left"
)

plt.tight_layout()

fig5_combined_png = FIGURE_DIR / "figure_5_combined_topic_distribution_ABC_revised.png"
fig5_combined_svg = FIGURE_DIR / "figure_5_combined_topic_distribution_ABC_revised.svg"

plt.savefig(fig5_combined_png, dpi=600, bbox_inches="tight")
plt.savefig(fig5_combined_svg, bbox_inches="tight")

plt.show()

print()
print("Saved revised combined Figure 5:")
print(fig5_combined_png)
print(fig5_combined_svg)

In [ ]:
# ============================================================
# CELL 36C — Export Figure 5 summary tables for manuscript writing
# Revised: no tabulate dependency
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# Load Table 4
# ------------------------------------------------------------
if "FINAL_TOPIC_DIR" not in globals():
    FINAL_TOPIC_DIR = BERTOPIC_DIR / "final_model"

table4_path = FINAL_TOPIC_DIR / "table4_final_bertopic_topics_revised.csv"

if not table4_path.exists():
    table4_path = FINAL_TOPIC_DIR / "table4_final_bertopic_topics.csv"

if not table4_path.exists():
    raise FileNotFoundError("Table 4 topic file was not found.")

table4 = pd.read_csv(table4_path, encoding="utf-8-sig")

table4["Topic ID"] = table4["Topic ID"].astype(int)
table4["Documents"] = pd.to_numeric(table4["Documents"], errors="coerce")
table4["Proportion of valid documents (%)"] = pd.to_numeric(
    table4["Proportion of valid documents (%)"],
    errors="coerce"
)

valid_document_total = table4["Documents"].sum()

# ------------------------------------------------------------
# Same broad theme grouping used for Figure 5
# ------------------------------------------------------------
theme_group_mapping = {
    0: "Infectious disease, virology, and AMR",
    1: "Parasitology and vector-borne disease",
    2: "Infectious disease, virology, and AMR",
    3: "Infectious disease, virology, and AMR",
    4: "Clinical medicine, diagnostics, and therapeutics",
    5: "Clinical medicine, diagnostics, and therapeutics",
    6: "Clinical medicine, diagnostics, and therapeutics",
    7: "Parasitology and vector-borne disease",
    8: "Veterinary public health and education",
    9: "Production animal health, nutrition, and physiology",
    10: "Infectious disease, virology, and AMR",
    11: "Production animal health, nutrition, and physiology",
    12: "Aquatic animal health",
    13: "Clinical medicine, diagnostics, and therapeutics",
    14: "Infectious disease, virology, and AMR",
    15: "Clinical medicine, diagnostics, and therapeutics",
    16: "Parasitology and vector-borne disease",
    17: "Clinical medicine, diagnostics, and therapeutics",
    18: "Clinical medicine, diagnostics, and therapeutics",
    19: "Clinical medicine, diagnostics, and therapeutics",
    20: "Infectious disease, virology, and AMR",
    21: "Production animal health, nutrition, and physiology",
    22: "Production animal health, nutrition, and physiology",
    23: "Infectious disease, virology, and AMR",
    24: "Infectious disease, virology, and AMR",
    25: "Clinical medicine, diagnostics, and therapeutics",
    26: "Production animal health, nutrition, and physiology",
    27: "Infectious disease, virology, and AMR",
    28: "Parasitology and vector-borne disease",
    29: "Clinical medicine, diagnostics, and therapeutics",
    30: "Reproduction and biotechnology",
    31: "Infectious disease, virology, and AMR",
    32: "Infectious disease, virology, and AMR",
    33: "Clinical medicine, diagnostics, and therapeutics",
    34: "Clinical medicine, diagnostics, and therapeutics",
    35: "Clinical medicine, diagnostics, and therapeutics",
    36: "Infectious disease, virology, and AMR",
    37: "Reproduction and biotechnology",
    38: "Parasitology and vector-borne disease",
    39: "Parasitology and vector-borne disease",
    40: "Parasitology and vector-borne disease",
    41: "Parasitology and vector-borne disease",
    42: "Infectious disease, virology, and AMR",
    43: "Clinical medicine, diagnostics, and therapeutics",
    44: "Parasitology and vector-borne disease",
    45: "Production animal health, nutrition, and physiology"
}

table4["Theme group"] = table4["Topic ID"].map(theme_group_mapping)

missing_theme_group = table4[table4["Theme group"].isna()]["Topic ID"].tolist()

if len(missing_theme_group) > 0:
    raise ValueError(f"Missing theme group mapping for Topic IDs: {missing_theme_group}")

# ------------------------------------------------------------
# Figure 5A/B summary table
# ------------------------------------------------------------
figure5_theme_summary = (
    table4
    .groupby("Theme group", as_index=False)
    .agg(
        Number_of_topics=("Topic ID", "nunique"),
        Documents=("Documents", "sum")
    )
)

figure5_theme_summary["Proportion of valid documents (%)"] = (
    figure5_theme_summary["Documents"] / valid_document_total * 100
)

figure5_theme_summary = (
    figure5_theme_summary
    .sort_values("Documents", ascending=False)
    .reset_index(drop=True)
)

figure5_theme_summary.insert(
    0,
    "Rank",
    np.arange(1, len(figure5_theme_summary) + 1)
)

figure5_theme_summary["Proportion of valid documents (%)"] = (
    figure5_theme_summary["Proportion of valid documents (%)"].round(2)
)

# ------------------------------------------------------------
# Figure 5C top 15 topic table
# ------------------------------------------------------------
figure5_top15_topics = (
    table4
    .sort_values("Documents", ascending=False)
    .head(15)
    [
        [
            "Rank",
            "Topic ID",
            "Topic label",
            "Documents",
            "Proportion of valid documents (%)",
            "Representative keywords"
        ]
    ]
    .copy()
)

# ------------------------------------------------------------
# Display tables
# ------------------------------------------------------------
print("Figure 5A/B data: broad thematic summary")
display(figure5_theme_summary)

print()
print("Figure 5C data: top 15 individual topics")
display(figure5_top15_topics)

# ------------------------------------------------------------
# Save as CSV files
# ------------------------------------------------------------
FIGURE5_TABLE_DIR = RESULTS_DIR / "tables"
FIGURE5_TABLE_DIR.mkdir(parents=True, exist_ok=True)

theme_summary_path = FIGURE5_TABLE_DIR / "figure5_theme_summary_for_manuscript.csv"
top15_topics_path = FIGURE5_TABLE_DIR / "figure5_top15_topics_for_manuscript.csv"

figure5_theme_summary.to_csv(
    theme_summary_path,
    index=False,
    encoding="utf-8-sig"
)

figure5_top15_topics.to_csv(
    top15_topics_path,
    index=False,
    encoding="utf-8-sig"
)

print()
print("Saved:")
print(theme_summary_path)
print(top15_topics_path)

# ------------------------------------------------------------
# Custom markdown-like table printer
# This does not require tabulate.
# ------------------------------------------------------------
def print_simple_table(df, title):
    print()
    print("=" * 80)
    print(title)
    print("=" * 80)
    
    df_print = df.copy()
    
    # Convert all values to strings
    for col in df_print.columns:
        df_print[col] = df_print[col].astype(str)
    
    # Column widths
    col_widths = {}
    for col in df_print.columns:
        max_value_width = df_print[col].map(len).max()
        col_widths[col] = max(len(col), max_value_width)
    
    # Header
    header = " | ".join(
        col.ljust(col_widths[col])
        for col in df_print.columns
    )
    
    separator = " | ".join(
        "-" * col_widths[col]
        for col in df_print.columns
    )
    
    print(header)
    print(separator)
    
    # Rows
    for _, row in df_print.iterrows():
        row_text = " | ".join(
            str(row[col]).ljust(col_widths[col])
            for col in df_print.columns
        )
        print(row_text)

# ------------------------------------------------------------
# Print copy-paste tables
# ------------------------------------------------------------
print_simple_table(
    figure5_theme_summary,
    "COPY THIS: Figure 5A/B broad theme summary"
)

print_simple_table(
    figure5_top15_topics,
    "COPY THIS: Figure 5C top 15 topics"
)

In [ ]:
# ============================================================
# CELL 37 — Figure 6
# Temporal evolution of BERTopic-derived research themes
#
# Figure 6A: Annual proportion of broad thematic categories
# Figure 6B: Annual heatmap of top 15 individual BERTopic topics
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
import textwrap
import math

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
if "FINAL_TOPIC_DIR" not in globals():
    FINAL_TOPIC_DIR = BERTOPIC_DIR / "final_model"

FIGURE_DIR = RESULTS_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TABLE_DIR = RESULTS_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

assignment_path = FINAL_TOPIC_DIR / "final_document_topic_assignments.csv"

table4_path = FINAL_TOPIC_DIR / "table4_final_bertopic_topics_revised.csv"

if not table4_path.exists():
    table4_path = FINAL_TOPIC_DIR / "table4_final_bertopic_topics.csv"

if not assignment_path.exists():
    raise FileNotFoundError(
        "final_document_topic_assignments.csv was not found."
    )

if not table4_path.exists():
    raise FileNotFoundError(
        "Table 4 topic file was not found."
    )

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------
topic_assignments = pd.read_csv(
    assignment_path,
    encoding="utf-8-sig",
    low_memory=False
)

table4 = pd.read_csv(
    table4_path,
    encoding="utf-8-sig",
    low_memory=False
)

print("Loaded:")
print(assignment_path)
print(table4_path)

# ------------------------------------------------------------
# Check required columns
# ------------------------------------------------------------
required_assignment_cols = ["Year", "Topic"]
required_table4_cols = [
    "Topic ID",
    "Topic label",
    "Documents",
    "Proportion of valid documents (%)"
]

missing_assignment_cols = [
    col for col in required_assignment_cols
    if col not in topic_assignments.columns
]

missing_table4_cols = [
    col for col in required_table4_cols
    if col not in table4.columns
]

if len(missing_assignment_cols) > 0:
    raise ValueError(f"Missing columns in topic assignment file: {missing_assignment_cols}")

if len(missing_table4_cols) > 0:
    raise ValueError(f"Missing columns in Table 4 file: {missing_table4_cols}")

# ------------------------------------------------------------
# Numeric formatting
# ------------------------------------------------------------
topic_assignments["Year"] = pd.to_numeric(
    topic_assignments["Year"],
    errors="coerce"
)

topic_assignments["Topic"] = pd.to_numeric(
    topic_assignments["Topic"],
    errors="coerce"
)

topic_assignments = topic_assignments.dropna(
    subset=["Year", "Topic"]
).copy()

topic_assignments["Year"] = topic_assignments["Year"].astype(int)
topic_assignments["Topic"] = topic_assignments["Topic"].astype(int)

table4["Topic ID"] = table4["Topic ID"].astype(int)
table4["Documents"] = pd.to_numeric(
    table4["Documents"],
    errors="coerce"
)

table4["Proportion of valid documents (%)"] = pd.to_numeric(
    table4["Proportion of valid documents (%)"],
    errors="coerce"
)

# ------------------------------------------------------------
# Study years
# ------------------------------------------------------------
year_min = int(topic_assignments["Year"].min())
year_max = int(topic_assignments["Year"].max())

years = list(range(year_min, year_max + 1))

print()
print("Year range:")
print(f"{year_min}–{year_max}")

# ------------------------------------------------------------
# Exclude Topic -1 outliers
# ------------------------------------------------------------
valid_assignments = topic_assignments[
    topic_assignments["Topic"] != -1
].copy()

print()
print("Document-topic assignment summary")
print("---------------------------------")
print(f"All BERTopic documents:           {len(topic_assignments):,}")
print(f"Valid topic-assigned documents:   {len(valid_assignments):,}")
print(f"Outlier documents:                {len(topic_assignments) - len(valid_assignments):,}")
print(f"Outlier ratio:                    {(len(topic_assignments) - len(valid_assignments)) / len(topic_assignments):.4f}")

# ------------------------------------------------------------
# Annual denominator
# Valid topic-assigned documents per year
# ------------------------------------------------------------
annual_valid_counts = (
    valid_assignments
    .groupby("Year")
    .size()
    .reindex(years, fill_value=0)
    .reset_index(name="Annual valid documents")
)

display(annual_valid_counts)

annual_valid_counts.to_csv(
    TABLE_DIR / "figure6_annual_valid_topic_assigned_documents.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Topic label mapping
# ------------------------------------------------------------
topic_label_map = dict(
    zip(
        table4["Topic ID"],
        table4["Topic label"]
    )
)

valid_assignments["Topic label"] = valid_assignments["Topic"].map(
    topic_label_map
)

missing_topic_labels = valid_assignments[
    valid_assignments["Topic label"].isna()
]["Topic"].unique().tolist()

if len(missing_topic_labels) > 0:
    raise ValueError(
        f"Missing topic labels for Topic IDs: {missing_topic_labels}"
    )

# ------------------------------------------------------------
# Broad theme group mapping
# Same grouping as Figure 5
# ------------------------------------------------------------
theme_group_mapping = {
    0: "Infectious disease, virology, and AMR",
    1: "Parasitology and vector-borne disease",
    2: "Infectious disease, virology, and AMR",
    3: "Infectious disease, virology, and AMR",
    4: "Clinical medicine, diagnostics, and therapeutics",
    5: "Clinical medicine, diagnostics, and therapeutics",
    6: "Clinical medicine, diagnostics, and therapeutics",
    7: "Parasitology and vector-borne disease",
    8: "Veterinary public health and education",
    9: "Production animal health, nutrition, and physiology",
    10: "Infectious disease, virology, and AMR",
    11: "Production animal health, nutrition, and physiology",
    12: "Aquatic animal health",
    13: "Clinical medicine, diagnostics, and therapeutics",
    14: "Infectious disease, virology, and AMR",
    15: "Clinical medicine, diagnostics, and therapeutics",
    16: "Parasitology and vector-borne disease",
    17: "Clinical medicine, diagnostics, and therapeutics",
    18: "Clinical medicine, diagnostics, and therapeutics",
    19: "Clinical medicine, diagnostics, and therapeutics",
    20: "Infectious disease, virology, and AMR",
    21: "Production animal health, nutrition, and physiology",
    22: "Production animal health, nutrition, and physiology",
    23: "Infectious disease, virology, and AMR",
    24: "Infectious disease, virology, and AMR",
    25: "Clinical medicine, diagnostics, and therapeutics",
    26: "Production animal health, nutrition, and physiology",
    27: "Infectious disease, virology, and AMR",
    28: "Parasitology and vector-borne disease",
    29: "Clinical medicine, diagnostics, and therapeutics",
    30: "Reproduction and biotechnology",
    31: "Infectious disease, virology, and AMR",
    32: "Infectious disease, virology, and AMR",
    33: "Clinical medicine, diagnostics, and therapeutics",
    34: "Clinical medicine, diagnostics, and therapeutics",
    35: "Clinical medicine, diagnostics, and therapeutics",
    36: "Infectious disease, virology, and AMR",
    37: "Reproduction and biotechnology",
    38: "Parasitology and vector-borne disease",
    39: "Parasitology and vector-borne disease",
    40: "Parasitology and vector-borne disease",
    41: "Parasitology and vector-borne disease",
    42: "Infectious disease, virology, and AMR",
    43: "Clinical medicine, diagnostics, and therapeutics",
    44: "Parasitology and vector-borne disease",
    45: "Production animal health, nutrition, and physiology"
}

valid_assignments["Theme group"] = valid_assignments["Topic"].map(
    theme_group_mapping
)

missing_theme_groups = valid_assignments[
    valid_assignments["Theme group"].isna()
]["Topic"].unique().tolist()

if len(missing_theme_groups) > 0:
    raise ValueError(
        f"Missing theme group mapping for Topic IDs: {missing_theme_groups}"
    )

# ------------------------------------------------------------
# Figure style
# ------------------------------------------------------------
plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.unicode_minus"] = False

actual_font = font_manager.findfont("Arial", fallback_to_default=True)
print()
print("Actual font used:")
print(actual_font)

def wrap_label(text, width=38):
    return "\n".join(textwrap.wrap(str(text), width=width))

def get_nice_ymax(max_value, tick_interval):
    return math.ceil(max_value / tick_interval) * tick_interval

def style_axis(ax, grid_axis="y"):
    ax.tick_params(
        axis="both",
        direction="out",
        length=4,
        width=0.9,
        labelsize=10
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.9)
    ax.spines["bottom"].set_linewidth(0.9)

    ax.grid(
        axis=grid_axis,
        linestyle="-",
        linewidth=0.4,
        alpha=0.25
    )

    ax.set_axisbelow(True)

# ============================================================
# Figure 6A data — Annual broad theme proportion
# ============================================================

annual_theme_counts = (
    valid_assignments
    .groupby(["Year", "Theme group"])
    .size()
    .reset_index(name="Documents")
)

annual_theme_counts = annual_theme_counts.merge(
    annual_valid_counts,
    on="Year",
    how="left"
)

annual_theme_counts["Annual proportion (%)"] = (
    annual_theme_counts["Documents"] /
    annual_theme_counts["Annual valid documents"] *
    100
)

theme_order = [
    "Infectious disease, virology, and AMR",
    "Clinical medicine, diagnostics, and therapeutics",
    "Parasitology and vector-borne disease",
    "Production animal health, nutrition, and physiology",
    "Veterinary public health and education",
    "Aquatic animal health",
    "Reproduction and biotechnology"
]

annual_theme_pivot = (
    annual_theme_counts
    .pivot_table(
        index="Year",
        columns="Theme group",
        values="Annual proportion (%)",
        aggfunc="sum"
    )
    .reindex(index=years, columns=theme_order)
    .fillna(0)
)

annual_theme_pivot.to_csv(
    TABLE_DIR / "figure6a_annual_broad_theme_proportion.csv",
    encoding="utf-8-sig"
)

display(annual_theme_pivot.round(2))

# ------------------------------------------------------------
# Theme temporal summary for manuscript interpretation
# ------------------------------------------------------------
theme_temporal_summary = []

for theme in theme_order:
    series = annual_theme_pivot[theme]

    peak_year = int(series.idxmax())
    peak_share = float(series.max())

    theme_temporal_summary.append({
        "Theme group": theme,
        "Share in first year (%)": round(float(series.loc[year_min]), 2),
        "Share in final year (%)": round(float(series.loc[year_max]), 2),
        "Change (% points)": round(float(series.loc[year_max] - series.loc[year_min]), 2),
        "Peak year": peak_year,
        "Peak share (%)": round(peak_share, 2)
    })

theme_temporal_summary = pd.DataFrame(theme_temporal_summary)

theme_temporal_summary.to_csv(
    TABLE_DIR / "figure6a_theme_temporal_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(theme_temporal_summary)

# ============================================================
# Figure 6A — Broad theme temporal line plot
# ============================================================

fig, ax = plt.subplots(figsize=(7, 4.5))

theme_colors = {
    "Infectious disease, virology, and AMR": "#C44E80",
    "Clinical medicine, diagnostics, and therapeutics": "#6D6D6D",
    "Parasitology and vector-borne disease": "#8172B3",
    "Production animal health, nutrition, and physiology": "#DD8452",
    "Veterinary public health and education": "#55A868",
    "Aquatic animal health": "#4C72B0",
    "Reproduction and biotechnology": "#937860"
}

for theme in theme_order:
    ax.plot(
        annual_theme_pivot.index,
        annual_theme_pivot[theme],
        marker="o",
        markersize=4.5,
        markerfacecolor="white",
        markeredgewidth=1.0,
        linewidth=1.3,
        color=theme_colors[theme],
        label=theme
    )

ax.set_xlim(year_min - 0.3, year_max + 0.3)

ax.set_xticks(years)
ax.set_xticklabels(
    years,
    rotation=45,
    ha="right",
    fontsize=9
)

theme_ymax = get_nice_ymax(
    annual_theme_pivot.max().max(),
    5
)

ax.set_ylim(0, theme_ymax + 3)
ax.yaxis.set_major_locator(ticker.MultipleLocator(5))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"{x:.0f}"))

ax.set_ylabel(
    "Annual proportion of valid documents (%)",
    fontsize=13,
    fontweight="bold"
)

ax.set_xlabel(
    "Year",
    fontsize=13,
    fontweight="bold"
)

style_axis(ax, grid_axis="y")

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.32),
    ncol=2,
    frameon=False,
    fontsize=8.2,
    handlelength=2.0,
    columnspacing=1.2
)

plt.tight_layout()

fig6a_png = FIGURE_DIR / "figure_6a_temporal_broad_theme_proportion.png"
fig6a_svg = FIGURE_DIR / "figure_6a_temporal_broad_theme_proportion.svg"

plt.savefig(fig6a_png, dpi=600, bbox_inches="tight")
plt.savefig(fig6a_svg, bbox_inches="tight")

plt.show()

print()
print("Saved Figure 6A:")
print(fig6a_png)
print(fig6a_svg)

# ============================================================
# Figure 6B data — Annual top 15 topic proportion heatmap
# ============================================================

top_n_topics = 15

top15_topics = (
    table4
    .sort_values("Documents", ascending=False)
    .head(top_n_topics)
    .copy()
)

top15_topic_ids = top15_topics["Topic ID"].tolist()

top15_label_map = dict(
    zip(
        top15_topics["Topic ID"],
        top15_topics["Topic label"]
    )
)

annual_topic_counts = (
    valid_assignments[
        valid_assignments["Topic"].isin(top15_topic_ids)
    ]
    .groupby(["Year", "Topic"])
    .size()
    .reset_index(name="Documents")
)

annual_topic_counts = annual_topic_counts.merge(
    annual_valid_counts,
    on="Year",
    how="left"
)

annual_topic_counts["Annual proportion (%)"] = (
    annual_topic_counts["Documents"] /
    annual_topic_counts["Annual valid documents"] *
    100
)

annual_topic_pivot = (
    annual_topic_counts
    .pivot_table(
        index="Topic",
        columns="Year",
        values="Annual proportion (%)",
        aggfunc="sum"
    )
    .reindex(index=top15_topic_ids, columns=years)
    .fillna(0)
)

annual_topic_pivot.to_csv(
    TABLE_DIR / "figure6b_annual_top15_topic_proportion.csv",
    encoding="utf-8-sig"
)

display(annual_topic_pivot.round(2))

# ------------------------------------------------------------
# Top 15 topic temporal summary for manuscript interpretation
# ------------------------------------------------------------
top15_temporal_summary = []

for topic_id in top15_topic_ids:
    series = annual_topic_pivot.loc[topic_id]

    peak_year = int(series.idxmax())
    peak_share = float(series.max())

    top15_temporal_summary.append({
        "Topic ID": topic_id,
        "Topic label": top15_label_map[topic_id],
        "Overall documents": int(
            table4.loc[table4["Topic ID"] == topic_id, "Documents"].iloc[0]
        ),
        "Overall proportion of valid documents (%)": round(
            float(
                table4.loc[
                    table4["Topic ID"] == topic_id,
                    "Proportion of valid documents (%)"
                ].iloc[0]
            ),
            2
        ),
        "Share in first year (%)": round(float(series.loc[year_min]), 2),
        "Share in final year (%)": round(float(series.loc[year_max]), 2),
        "Change (% points)": round(float(series.loc[year_max] - series.loc[year_min]), 2),
        "Peak year": peak_year,
        "Peak share (%)": round(peak_share, 2)
    })

top15_temporal_summary = pd.DataFrame(top15_temporal_summary)

top15_temporal_summary.to_csv(
    TABLE_DIR / "figure6b_top15_topic_temporal_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(top15_temporal_summary)

# ============================================================
# Figure 6B — Heatmap of top 15 topic temporal proportions
# ============================================================

fig, ax = plt.subplots(figsize=(7, 6))

heatmap_data = annual_topic_pivot.values

heatmap_cmap = LinearSegmentedColormap.from_list(
    "custom_pink_heatmap",
    ["#F7F7F7", "#F4C2D7", "#D75F93", "#8E2F5D"]
)

im = ax.imshow(
    heatmap_data,
    aspect="auto",
    cmap=heatmap_cmap,
    interpolation="nearest"
)

ax.set_xticks(np.arange(len(years)))
ax.set_xticklabels(
    years,
    rotation=45,
    ha="right",
    fontsize=9
)

topic_y_labels = [
    wrap_label(f"T{topic_id}: {top15_label_map[topic_id]}", width=42)
    for topic_id in annual_topic_pivot.index
]

ax.set_yticks(np.arange(len(topic_y_labels)))
ax.set_yticklabels(
    topic_y_labels,
    fontsize=8.5
)

ax.set_xlabel(
    "Year",
    fontsize=13,
    fontweight="bold"
)

ax.set_ylabel(
    "BERTopic-derived topic",
    fontsize=13,
    fontweight="bold"
)

ax.tick_params(
    axis="both",
    direction="out",
    length=3,
    width=0.8
)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)

# Horizontal colorbar
cbar = fig.colorbar(
    im,
    ax=ax,
    orientation="horizontal",
    pad=0.14,
    fraction=0.05
)

cbar.set_label(
    "Annual proportion of valid documents (%)",
    fontsize=11,
    fontweight="bold"
)

cbar.ax.tick_params(
    labelsize=9,
    direction="out",
    length=3,
    width=0.8
)

plt.tight_layout()

fig6b_png = FIGURE_DIR / "figure_6b_temporal_top15_topic_heatmap.png"
fig6b_svg = FIGURE_DIR / "figure_6b_temporal_top15_topic_heatmap.svg"

plt.savefig(fig6b_png, dpi=600, bbox_inches="tight")
plt.savefig(fig6b_svg, bbox_inches="tight")

plt.show()

print()
print("Saved Figure 6B:")
print(fig6b_png)
print(fig6b_svg)

# ============================================================
# Combined Figure 6 — A/B
# ============================================================

fig = plt.figure(figsize=(7.4, 9.4))

gs = fig.add_gridspec(
    nrows=2,
    ncols=1,
    height_ratios=[1.0, 1.55],
    hspace=0.52
)

# ------------------------------------------------------------
# Panel A
# ------------------------------------------------------------
ax1 = fig.add_subplot(gs[0, 0])

for theme in theme_order:
    ax1.plot(
        annual_theme_pivot.index,
        annual_theme_pivot[theme],
        marker="o",
        markersize=4.0,
        markerfacecolor="white",
        markeredgewidth=1.0,
        linewidth=1.2,
        color=theme_colors[theme],
        label=theme
    )

ax1.set_xlim(year_min - 0.3, year_max + 0.3)

ax1.set_xticks(years)
ax1.set_xticklabels(
    years,
    rotation=45,
    ha="right",
    fontsize=8.5
)

ax1.set_ylim(0, theme_ymax + 3)
ax1.yaxis.set_major_locator(ticker.MultipleLocator(5))
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"{x:.0f}"))

ax1.set_ylabel(
    "Annual proportion (%)",
    fontsize=11,
    fontweight="bold"
)

ax1.set_xlabel(
    "Year",
    fontsize=11,
    fontweight="bold"
)

style_axis(ax1, grid_axis="y")

ax1.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.42),
    ncol=2,
    frameon=False,
    fontsize=7.8,
    handlelength=1.8,
    columnspacing=1.0
)

ax1.text(
    -0.13,
    1.05,
    "A",
    transform=ax1.transAxes,
    fontsize=13,
    fontweight="bold",
    va="bottom",
    ha="left"
)

# ------------------------------------------------------------
# Panel B
# ------------------------------------------------------------
ax2 = fig.add_subplot(gs[1, 0])

im2 = ax2.imshow(
    heatmap_data,
    aspect="auto",
    cmap=heatmap_cmap,
    interpolation="nearest"
)

ax2.set_xticks(np.arange(len(years)))
ax2.set_xticklabels(
    years,
    rotation=45,
    ha="right",
    fontsize=8.5
)

ax2.set_yticks(np.arange(len(topic_y_labels)))
ax2.set_yticklabels(
    topic_y_labels,
    fontsize=7.6
)

ax2.set_xlabel(
    "Year",
    fontsize=11,
    fontweight="bold"
)

ax2.set_ylabel(
    "BERTopic-derived topic",
    fontsize=11,
    fontweight="bold"
)

ax2.tick_params(
    axis="both",
    direction="out",
    length=3,
    width=0.8
)

for spine in ax2.spines.values():
    spine.set_linewidth(0.8)

cbar2 = fig.colorbar(
    im2,
    ax=ax2,
    orientation="horizontal",
    pad=0.13,
    fraction=0.045
)

cbar2.set_label(
    "Annual proportion of valid documents (%)",
    fontsize=10,
    fontweight="bold"
)

cbar2.ax.tick_params(
    labelsize=8.5,
    direction="out",
    length=3,
    width=0.8
)

ax2.text(
    -0.13,
    1.03,
    "B",
    transform=ax2.transAxes,
    fontsize=13,
    fontweight="bold",
    va="bottom",
    ha="left"
)

plt.tight_layout()

fig6_combined_png = FIGURE_DIR / "figure_6_temporal_evolution_AB.png"
fig6_combined_svg = FIGURE_DIR / "figure_6_temporal_evolution_AB.svg"

plt.savefig(fig6_combined_png, dpi=600, bbox_inches="tight")
plt.savefig(fig6_combined_svg, bbox_inches="tight")

plt.show()

print()
print("Saved combined Figure 6:")
print(fig6_combined_png)
print(fig6_combined_svg)

# ============================================================
# Print copy-paste summaries for manuscript writing
# ============================================================

def print_simple_table(df, title):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)

    df_print = df.copy()

    for col in df_print.columns:
        df_print[col] = df_print[col].astype(str)

    col_widths = {}

    for col in df_print.columns:
        max_value_width = df_print[col].map(len).max()
        col_widths[col] = max(len(col), max_value_width)

    header = " | ".join(
        col.ljust(col_widths[col])
        for col in df_print.columns
    )

    separator = " | ".join(
        "-" * col_widths[col]
        for col in df_print.columns
    )

    print(header)
    print(separator)

    for _, row in df_print.iterrows():
        row_text = " | ".join(
            str(row[col]).ljust(col_widths[col])
            for col in df_print.columns
        )
        print(row_text)

print_simple_table(
    theme_temporal_summary,
    "COPY THIS: Figure 6A broad theme temporal summary"
)

print_simple_table(
    top15_temporal_summary,
    "COPY THIS: Figure 6B top 15 topic temporal summary"
)

In [ ]:
# ============================================================
# Check software and package versions for manuscript Section 2.10
# ============================================================

import sys
import importlib.metadata as md

packages = [
    "pandas",
    "numpy",
    "scikit-learn",
    "networkx",
    "matplotlib",
    "geopandas",
    "sentence-transformers",
    "bertopic",
    "umap-learn",
    "hdbscan",
]

print("Python version")
print("--------------")
print(sys.version.split()[0])
print()

print("Package versions")
print("----------------")
for package in packages:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: not installed / not found")